In [ ]:
# ============================================================
# IROS-READY — NeuroCommitSSM (SSL + FT) + Commit-as-Decision API
# UPDATED (training/fusion/regularization/features/weights)
# ------------------------------------------------------------
# ============================================================

from __future__ import annotations

from pathlib import Path
import os
os.environ.setdefault("OMP_NUM_THREADS", "8")
os.environ.setdefault("MKL_NUM_THREADS", "8")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from dataclasses import dataclass, asdict
from typing import Dict, Any, Optional, Tuple, List
import math
import time
import random
import json
from collections import OrderedDict, defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.backends.cudnn as cudnn
import torch.multiprocessing as tmp
from torch.utils.checkpoint import checkpoint as _checkpoint
from contextlib import nullcontext
import warnings
warnings.filterwarnings("ignore", message="`torch.backends.cuda.sdp_kernel()` is deprecated")


# ---- FIX cuDNN/SDPA plan errors + nvrtc warnings ----
if torch.cuda.is_available():
    try:
        torch.backends.cuda.enable_flash_sdp(False)
        torch.backends.cuda.enable_mem_efficient_sdp(False)
        torch.backends.cuda.enable_math_sdp(True)
    except Exception:
        pass

# ---- HARD FORCE: use only math SDPA (avoid cuDNN/flash/mem-eff) ----


def sdp_math_only():
    if not torch.cuda.is_available():
        return nullcontext()
    try:
        from torch.nn.attention import sdpa_kernel, SDPBackend
        return sdpa_kernel([SDPBackend.MATH])
    except Exception:
        pass
    try:
        if hasattr(torch.backends.cuda, "sdp_kernel"):
            return torch.backends.cuda.sdp_kernel(
                enable_flash=False,
                enable_mem_efficient=False,
                enable_math=True,
            )
    except Exception:
        pass
    return nullcontext()


if torch.cuda.is_available() and hasattr(torch.backends.cuda, "enable_cudnn_sdp"):
    torch.backends.cuda.enable_cudnn_sdp(False)

# Speed-focused (not strict determinism)
cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ============================================================
# Paths / tags  (Phase 6)
# ============================================================
from pathlib import Path

DATA_ROOT  = Path("/home/tsultan1/IROS/Data/_dataset_icml_v1")

# ✅ Use commit-meta Phase-5 exports
SUP_EXP_PREFIX = "exports_v6_3c_commitmeta_balanced_core_ok"
SSL_EXP_PREFIX = "exports_v6_3c_commitmeta_ssl_core_ok"

# ✅ Match your NEW Phase-5.5 commitmeta feature filenames (exact)
# Example:
#  features_v6_3c_commitmeta_eegpsd_emgtd_mask_full_balanced_core_ok_fold35_train.npz
FEATURE_PATTERN = "features_v6_3c_commitmeta_eegpsd_emgtd_mask_full_balanced_core_ok_fold{fold}_{split}.npz"

ART_ROOT = Path("/home/tsultan1/IROS/artifacts")
TAG = "IROS_NeuroCommitSSM_CommitAsDecision_v1_run3"


AUTO_DETECT_FOLDS = True
FOLDS_EXPLICIT = list(range(1, 31)) + list(range(32, 39))

# ============================================================
# Global config
# ============================================================
SSL_ENABLE      = True
SSL_EPOCHS      = 30
SSL_LR          = 2e-4
SSL_WD          = 1e-2
SSL_BS_TRAIN    = 256
SSL_TEMP        = 0.20
SSL_LAMBDA_NCE  = 1.0
SSL_LAMBDA_REC  = 1.0
SSL_PATCH_SIZE  = 25
SSL_MASK_PATCH_FRAC = 0.25

EPOCHS     = 30
BS_TRAIN   = 128
BS_EVAL    = 512
NUM_WORKERS= 8
CACHE_SIZE = 16
BASE_SEED  = 42

# -------- Phase 5.5 features ----------
# Your features are: X_psd=(N,96), X_emg=(N,24), X_mask=(N,3) => total=123
USE_P55_FEATURES = "auto"   # "auto" | True | False

# Loss weights
ALPHA_TASK        = 1.0
USE_SAMPLE_WEIGHT = True     # ✅ turn on

USE_ACTION_SAMPLER = True
SAMPLER_MODE       = "shard_aware"  # "shard_aware" | "none"

# Action threshold tuning (for ACTION head)
THR_OBJECTIVE    = "macroF1"      # "macroF1" or "bal_acc"
THR_SEARCH_STEPS = 61

# Commit decision filter metrics assume stride (your Phase-3 window stride is 0.25s)
WIN_STRIDE_S = 0.25

DO_R3R4_EVAL = False
SAVE_PREDS   = False

USE_AMP = True
USE_GRADIENT_CHECKPOINTING = True

ALL_SCENARIOS = ["S0","S1","S2","S3","S4","S5","S6"]

# ============================================================
# ✅ NEW: Train like you evaluate (multi-scenario training)
# ============================================================
TRAIN_SCENARIO_MIX = True
P_SCEN_TRAIN = 0.70  # apply S0..S3
SCEN_TRAIN_CHOICES = ["S0","S1","S2","S3"]
# emphasize S2 (no EMG) because it's your weak point
SCEN_TRAIN_W = [0.35, 0.20, 0.30, 0.15]

P_DROP2_TRAIN = 0.05
 # sometimes force unimodal
DROP2_CHOICES = ["S4","S5","S6"]

# ============================================================
# ✅ NEW: Dominance regularizer (prevents EMG takeover)
# ============================================================
DOM_REG_ENABLE = True
DOM_MIN_ENTROPY = 0.65
DOM_REG_WEIGHT  = 0.20



print("IROS CONFIG LOADED:")
print(" DATA_ROOT:", DATA_ROOT)
print(" SUP_EXP_PREFIX:", SUP_EXP_PREFIX)
print(" SSL_EXP_PREFIX:", SSL_EXP_PREFIX)
print(" ART_ROOT:", ART_ROOT)
print(" TAG:", TAG)
print(f" NUM_WORKERS={NUM_WORKERS} | CACHE_SIZE={CACHE_SIZE} | AMP={USE_AMP} | CKPT={USE_GRADIENT_CHECKPOINTING}")
print(f" SSL_EPOCHS={SSL_EPOCHS} | FT_EPOCHS={EPOCHS} | THR_SEARCH_STEPS={THR_SEARCH_STEPS}")
print(f" ALL_SCENARIOS={ALL_SCENARIOS}")
print(f" WIN_STRIDE_S={WIN_STRIDE_S}")
print(f" TRAIN_SCENARIO_MIX={TRAIN_SCENARIO_MIX} P_SCEN_TRAIN={P_SCEN_TRAIN} P_DROP2_TRAIN={P_DROP2_TRAIN}")
print(f" DOM_REG_ENABLE={DOM_REG_ENABLE} DOM_MIN_ENTROPY={DOM_MIN_ENTROPY} DOM_REG_WEIGHT={DOM_REG_WEIGHT}")
print(f" USE_SAMPLE_WEIGHT={USE_SAMPLE_WEIGHT}")

# ============================================================
# Utils
# ============================================================
def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    cudnn.deterministic = False
    cudnn.benchmark = True

def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p

def to_device(batch: Dict[str, Any], device: str, skip_keys: Optional[set] = None) -> Dict[str, Any]:
    skip_keys = skip_keys or set()
    out = {}
    for k, v in batch.items():
        if k in skip_keys:
            out[k] = v
        elif torch.is_tensor(v):
            out[k] = v.to(device, non_blocking=True)
        else:
            out[k] = v
    return out

def count_params(model: torch.nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

@torch.no_grad()
def measure_infer_latency_ms(model, sample_batch, device: str, iters: int = 200, warmup: int = 20, use_amp: bool = True):
    model.eval()
    dev_type = "cuda" if str(device).startswith("cuda") else "cpu"
    b = to_device(sample_batch, device)

    for _ in range(warmup):
        with torch.autocast(device_type=dev_type, enabled=bool(use_amp and dev_type == "cuda")):
            _ = model.forward_window(b)
    if dev_type == "cuda":
        torch.cuda.synchronize()

    t0 = time.perf_counter()
    for _ in range(iters):
        with torch.autocast(device_type=dev_type, enabled=bool(use_amp and dev_type == "cuda")):
            _ = model.forward_window(b)
    if dev_type == "cuda":
        torch.cuda.synchronize()
    t1 = time.perf_counter()

    ms = (t1 - t0) * 1000.0 / max(1, iters)
    return float(ms)

def save_runtime_json(out_dir: Path, fold: int, runtime: dict):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    path = out_dir / f"runtime_fold{fold}.json"
    with open(path, "w") as f:
        json.dump(runtime, f, indent=2)
    print("[done] wrote:", path)

# Keep only one build_task_mask
def build_task_mask(batch: Dict[str, torch.Tensor]) -> torch.Tensor:
    ya = batch["y_action"].long()
    tv = batch.get("task_valid", torch.ones_like(ya)).long()
    uft = batch.get("use_for_task", torch.ones_like(ya)).long()
    yt = batch.get("y_task_raw", torch.zeros_like(ya)).long()
    return ((ya == 1) & (tv == 1) & (uft == 1) & (yt > 0))

@torch.no_grad()
def confusion_matrix_np(y_true, y_pred, n_cls: int):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    idx = n_cls * y_true + y_pred
    cm = np.bincount(idx, minlength=n_cls*n_cls).reshape(n_cls, n_cls)
    return cm

@torch.no_grad()
def classification_report_np(y_true, y_pred, class_names: List[str]):
    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    n_cls = len(class_names)
    cm = confusion_matrix_np(y_true, y_pred, n_cls)

    support = cm.sum(axis=1).astype(np.int64)
    tp = np.diag(cm).astype(np.float64)
    fp = cm.sum(axis=0).astype(np.float64) - tp
    fn = cm.sum(axis=1).astype(np.float64) - tp
    tn = cm.sum().astype(np.float64) - (tp + fp + fn)

    precision = tp / (tp + fp + 1e-12)
    recall    = tp / (tp + fn + 1e-12)
    f1        = (2 * precision * recall) / (precision + recall + 1e-12)
    acc_cls   = tp / (support + 1e-12)
    specificity = tn / (tn + fp + 1e-12)

    acc = float(tp.sum() / max(1, cm.sum()))
    present = support > 0
    bal_acc = float(recall[present].mean()) if present.any() else 0.0
    macro_f1 = float(f1[present].mean()) if present.any() else 0.0
    macro_p  = float(precision[present].mean()) if present.any() else 0.0
    macro_r  = float(recall[present].mean()) if present.any() else 0.0

    rows = []
    for i, name in enumerate(class_names):
        rows.append({
            "class_id": int(i),
            "class_name": str(name),
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "acc_class": float(acc_cls[i]),
            "specificity": float(specificity[i]),
            "support": int(support[i]),
        })

    return {
        "acc": acc,
        "bal_acc": bal_acc,
        "macro_precision": macro_p,
        "macro_recall": macro_r,
        "macro_f1": macro_f1,
        "cm": cm,
        "rows": rows,
    }

def binary_average_precision(y_true: np.ndarray, y_score: np.ndarray) -> float:
    y_true = np.asarray(y_true).astype(np.int64)
    y_score = np.asarray(y_score).astype(np.float64)
    if y_true.size == 0:
        return 0.0
    pos = (y_true == 1)
    if pos.sum() == 0:
        return 0.0

    order = np.argsort(-y_score)
    y_true = y_true[order]
    pos = (y_true == 1)

    tp = np.cumsum(pos)
    fp = np.cumsum(~pos)

    precision = tp / (tp + fp + 1e-12)
    recall = tp / (pos.sum() + 1e-12)

    recall_prev = np.concatenate([[0.0], recall[:-1]])
    delta = recall - recall_prev
    ap = float(np.sum(precision * delta))
    return ap

def multiclass_macro_auprc_ovr(y_trueK: np.ndarray, prob: np.ndarray, num_class: int) -> float:
    y_trueK = np.asarray(y_trueK).astype(np.int64)
    prob = np.asarray(prob).astype(np.float64)
    if y_trueK.size == 0:
        return 0.0
    aps = []
    for k in range(num_class):
        yt = (y_trueK == k).astype(np.int64)
        if yt.sum() == 0:
            continue
        aps.append(binary_average_precision(yt, prob[:, k]))
    return float(np.mean(aps)) if len(aps) else 0.0

def rest_fp_rate_action(y_true01: np.ndarray, y_pred01: np.ndarray) -> float:
    y_true01 = np.asarray(y_true01).astype(np.int64)
    y_pred01 = np.asarray(y_pred01).astype(np.int64)
    m = (y_true01 == 0)
    denom = int(m.sum())
    if denom == 0:
        return 0.0
    return float(((y_pred01 == 1) & m).sum() / denom)

def detect_folds(dataset_dir: Path, exp_prefix: str) -> List[int]:
    folds = []
    for p in dataset_dir.glob(f"{exp_prefix}_fold*"):
        if not p.is_dir():
            continue
        suf = p.name.split("_fold")[-1]
        try:
            k = int(suf)
        except Exception:
            continue
        if all((p/s).exists() for s in ["train","val","test"]) and len(list((p/"train").glob("*_shard_*.npz"))) > 0:
            folds.append(k)
    return sorted(set(folds))

# ============================================================
# Scenarios + optional robustness (R3/R4)
# ============================================================
SCENARIO_KEEP = {
    "S0": (1, 1, 1),
    "S1": (0, 1, 1),
    "S2": (1, 0, 1),
    "S3": (1, 1, 0),
    "S4": (1, 0, 0),
    "S5": (0, 1, 0),
    "S6": (0, 0, 1),
}

def _zero_rel(x: torch.Tensor) -> torch.Tensor:
    return torch.zeros_like(x)

def apply_scenario_inplace(batch: Dict[str, torch.Tensor], scenario: str):
    ke, km, kt = SCENARIO_KEEP[scenario]

    if ke == 0:
        batch["X_EEG"].zero_(); batch["M_EEG"].zero_()
        if "r_eeg" in batch: batch["r_eeg"] = _zero_rel(batch["r_eeg"])
        if "F_psd" in batch: batch["F_psd"].zero_()
    if km == 0:
        batch["X_EMG"].zero_(); batch["M_EMG"].zero_()
        if "r_emg" in batch: batch["r_emg"] = _zero_rel(batch["r_emg"])
        if "F_emg" in batch: batch["F_emg"].zero_()
    if kt == 0:
        batch["X_ET"].zero_(); batch["M_ET"].zero_()
        if "r_et" in batch: batch["r_et"] = _zero_rel(batch["r_et"])

    if "F_mask" in batch:
        if ke == 0: batch["F_mask"][:, 0] = 0.0
        if km == 0: batch["F_mask"][:, 1] = 0.0
        if kt == 0: batch["F_mask"][:, 2] = 0.0

    return batch

def apply_train_scenario_mix_inplace(batch: Dict[str, torch.Tensor]):
    """
    FIXED: pick ONE scenario only (no sequential scenario application).
    This avoids 'all modalities zero' bugs.
    """
    if not TRAIN_SCENARIO_MIX:
        return batch

    r = random.random()

    # 1) sometimes force unimodal directly
    if r < float(P_DROP2_TRAIN):
        sc = random.choice(DROP2_CHOICES)   # S4/S5/S6
        return apply_scenario_inplace(batch, sc)

    # 2) sometimes apply S0..S3 mix
    if r < float(P_DROP2_TRAIN) + float(P_SCEN_TRAIN):
        sc = random.choices(SCEN_TRAIN_CHOICES, weights=SCEN_TRAIN_W, k=1)[0]
        return apply_scenario_inplace(batch, sc)

    # 3) otherwise leave as-is (effectively S0)
    return batch

def apply_R3_partial_time_dropout_inplace(batch: Dict[str, torch.Tensor], chunk_frac: float = 0.20, p: float = 1.0):
    if random.random() > p:
        return batch
    mod = random.choice(["EEG", "EMG", "ET"])
    Xk = f"X_{mod}"
    Mk = f"M_{mod}"
    X = batch[Xk]; M = batch[Mk]
    B, T, C = X.shape
    w = max(1, int(T * float(chunk_frac)))
    t0 = random.randint(0, max(0, T - w))
    X[:, t0:t0+w, :].zero_()
    M[:, t0:t0+w, :].zero_()
    rk_key = {"EEG": "r_eeg", "EMG": "r_emg", "ET": "r_et"}[mod]
    rk = batch.get(rk_key, None)
    if rk is not None:
        batch[rk_key] = 0.5 * rk
    return batch

def apply_R4_noise_artifact_inplace(batch: Dict[str, torch.Tensor], sigma: float = 0.05, p: float = 1.0):
    if random.random() > p:
        return batch
    for mod in ["EEG","EMG","ET"]:
        X = batch[f"X_{mod}"]; M = batch[f"M_{mod}"]
        X.add_(torch.randn_like(X) * float(sigma) * (M > 0).float())
        if random.random() < 0.35:
            B, T, C = X.shape
            w = max(1, int(T * 0.06))
            t0 = random.randint(0, max(0, T - w))
            X[:, t0:t0+w, :].zero_()
            M[:, t0:t0+w, :].zero_()
        batch[f"X_{mod}"] = X
        batch[f"M_{mod}"] = M
    return batch

# ============================================================
# Features auto-discovery
# ============================================================
def find_feature_file(dataset_dir: Path, fold: int, split: str, pattern: str) -> Optional[Path]:
    pat = pattern.format(fold=fold, split=split)
    cands = sorted(dataset_dir.glob(pat))
    if not cands:
        return None
    cands = sorted(cands, key=lambda p: p.stat().st_mtime, reverse=True)
    return cands[0]

# ============================================================
# Dataset + Sampler
# ============================================================
def _pad_or_trim_rel(v: np.ndarray, rel_dim: int) -> np.ndarray:
    v = np.asarray(v, dtype=np.float32).reshape(-1)
    if v.size == rel_dim:
        return v
    if v.size > rel_dim:
        return v[:rel_dim]
    out = np.zeros((rel_dim,), dtype=np.float32)
    out[:v.size] = v
    return out

class ShardNPZDataset(torch.utils.data.Dataset):
    """
    Window-level dataset (Phase exports compatible) + IROS commit supervision helpers.

    ✅ UPDATES (commit-meta ready):
      - Sample weighting enabled (shard sample_weight > feature sample_weight > q_score > 1.0)
      - Outputs F_psd/F_emg/F_mask when Phase-5.5 feature files exist
      - Commit supervision:
          * onset-based (commit_mask=1.0) requires (dist_to_onset_s finite) AND (onset_idx_nearest >= 0)
          * proxy-based (commit_mask=0.5) uses active_prob_mean or w_label_mean
          * NO weak fallback (commit_mask=0.0) — prevents ct inflation / false commits
    """
    def __init__(
        self,
        split_dir: Path,
        keep_frac: float = 1.0,
        seed: int = 42,
        feature_file: Optional[Path] = None,
        strict_feature_align: bool = True,
        build_action_index: bool = True,
        rel_dim: int = 4,
        cache_size: int = 16,
    ):
        super().__init__()
        self.split_dir = Path(split_dir)
        self.shard_paths = sorted(self.split_dir.glob("*_shard_*.npz"))
        if not self.shard_paths:
            raise FileNotFoundError(f"No shard npz found in {self.split_dir}")

        self.rel_dim = int(rel_dim)
        self.cache_size = int(max(1, cache_size))

        self.shard_sizes = []
        for sp in self.shard_paths:
            with np.load(sp, allow_pickle=False) as z:
                n = int(z["y_action"].shape[0]) if "y_action" in z.files else int(z["X_EEG"].shape[0])
            self.shard_sizes.append(n)

        self.cum = np.cumsum([0] + self.shard_sizes)
        self.N_total = int(self.cum[-1])

        idx = np.arange(self.N_total, dtype=np.int64)
        if keep_frac < 1.0:
            rng = np.random.RandomState(seed)
            rng.shuffle(idx)
            idx = np.sort(idx[:max(1, int(self.N_total * keep_frac))])
        self.idx_global = idx
        self.N = int(self.idx_global.shape[0])

        sid = np.searchsorted(self.cum, self.idx_global, side="right") - 1
        sid = sid.astype(np.int64)
        lid = (self.idx_global - self.cum[sid]).astype(np.int64)
        self.sid_of_i = sid
        self.lid_of_i = lid

        # features (optional)
        self.feature_file = feature_file
        self.has_features = feature_file is not None
        self.strict_feature_align = bool(strict_feature_align)
        self.F_has_sw = False
        if self.has_features:
            with np.load(feature_file, allow_pickle=False) as fz:
                self.F_X_psd  = fz["X_psd"]
                self.F_X_emg  = fz["X_emg"]
                self.F_X_mask = fz["X_mask"]
                self.F_has_sw = ("sample_weight" in fz.files)
                self.F_sample_weight = fz["sample_weight"] if self.F_has_sw else None

            if self.strict_feature_align:
                if int(self.F_X_psd.shape[0]) != int(self.N_total):
                    raise RuntimeError(
                        f"[features align] N mismatch: shards={self.N_total} vs feats={self.F_X_psd.shape[0]} file={feature_file}"
                    )

        # action labels (for sampler)
        self._y_action_all = None
        if build_action_index:
            y_list = []
            for sp in self.shard_paths:
                with np.load(sp, allow_pickle=False) as z:
                    if "y_action" not in z.files:
                        raise RuntimeError(f"Shard missing y_action: {sp}")
                    y_list.append(np.asarray(z["y_action"]).astype(np.int64, copy=False))
            self._y_action_all = np.concatenate(y_list, axis=0)
            if int(self._y_action_all.shape[0]) != int(self.N_total):
                raise RuntimeError("Internal error: y_action length mismatch")

        self._cache = OrderedDict()

    def __len__(self):
        return self.N

    def action_labels_subset(self) -> Optional[np.ndarray]:
        if self._y_action_all is None:
            return None
        return self._y_action_all[self.idx_global]

    def _load_shard(self, sid: int):
        if sid in self._cache:
            self._cache.move_to_end(sid)
            return self._cache[sid]

        path = self.shard_paths[sid]

        def _cast_key(k: str, arr: np.ndarray) -> np.ndarray:
            if k.startswith(("X_", "M_")):
                return arr.astype(np.float32, copy=False)
            if k.startswith("y_") or k in ("task_valid", "use_for_task"):
                return arr.astype(np.int64, copy=False)
            if k in ("sample_weight", "q_score", "dist_to_onset_s", "w_label_mean", "active_prob_mean", "center_time_s"):
                return arr.astype(np.float32, copy=False)
            if k.startswith(("r_", "p_")):
                return arr.astype(np.float32, copy=False)
            if k in ("is_transition", "onset_idx_nearest"):
                return arr.astype(np.int64, copy=False)
            if k in ("onset_time_nearest_s",):
                return arr.astype(np.float32, copy=False)
            return arr

        with np.load(path, allow_pickle=False) as z:
            data = {k: _cast_key(k, np.asarray(z[k])) for k in z.files}

        self._cache[sid] = data
        if len(self._cache) > self.cache_size:
            self._cache.popitem(last=False)
        return data

    def __getitem__(self, i: int):
        sid = int(self.sid_of_i[i])
        lid = int(self.lid_of_i[i])
        gidx = int(self.idx_global[i])

        z = self._load_shard(sid)

        X_EEG = z["X_EEG"][lid]; M_EEG = z["M_EEG"][lid]
        X_EMG = z["X_EMG"][lid]; M_EMG = z["M_EMG"][lid]
        X_ET  = z["X_ET"][lid];  M_ET  = z["M_ET"][lid]

        y_action = int(z["y_action"][lid])
        ya_int = int(y_action)

        if "y_task" in z:
            y_task = int(z["y_task"][lid])
        elif "y_task_raw" in z:
            y_task = int(z["y_task_raw"][lid])
        else:
            y_task = 0

        task_valid   = int(z["task_valid"][lid])   if "task_valid" in z else 1
        use_for_task = int(z["use_for_task"][lid]) if "use_for_task" in z else 1

        # sample_weight priority:
        # shard sample_weight > feature sample_weight > q_score > 1.0
        if "sample_weight" in z:
            sample_weight = float(z["sample_weight"][lid])
        elif self.has_features and self.F_has_sw:
            sample_weight = float(self.F_sample_weight[gidx])
        elif "q_score" in z:
            sample_weight = float(z["q_score"][lid])
        else:
            sample_weight = 1.0

        re = z["r_eeg"][lid] if "r_eeg" in z else np.ones((self.rel_dim,), np.float32)
        rm = z["r_emg"][lid] if "r_emg" in z else np.ones((self.rel_dim,), np.float32)
        rt = z["r_et"][lid]  if "r_et"  in z else np.ones((self.rel_dim,), np.float32)
        re = _pad_or_trim_rel(re, self.rel_dim)
        rm = _pad_or_trim_rel(rm, self.rel_dim)
        rt = _pad_or_trim_rel(rt, self.rel_dim)

        out: Dict[str, Any] = {
            "X_EEG": torch.from_numpy(X_EEG),
            "M_EEG": torch.from_numpy(M_EEG),
            "X_EMG": torch.from_numpy(X_EMG),
            "M_EMG": torch.from_numpy(M_EMG),
            "X_ET":  torch.from_numpy(X_ET),
            "M_ET":  torch.from_numpy(M_ET),

            "y_action": torch.tensor(y_action, dtype=torch.long),
            "y_task_raw": torch.tensor(y_task, dtype=torch.long),
            "task_valid": torch.tensor(task_valid, dtype=torch.long),
            "use_for_task": torch.tensor(use_for_task, dtype=torch.long),

            "sample_weight": torch.tensor(sample_weight, dtype=torch.float32),

            "r_eeg": torch.from_numpy(re),
            "r_emg": torch.from_numpy(rm),
            "r_et":  torch.from_numpy(rt),
        }

        if self.has_features:
            out["F_psd"]  = torch.from_numpy(np.asarray(self.F_X_psd[gidx], dtype=np.float32))
            out["F_emg"]  = torch.from_numpy(np.asarray(self.F_X_emg[gidx], dtype=np.float32))
            out["F_mask"] = torch.from_numpy(np.asarray(self.F_X_mask[gidx], dtype=np.float32))

        # safe metas (strings/ints/floats)
        meta_keys = [
            "subject_id",
            "task_code", "task",
            "trial_id", "trial",
            "win_type", "win_idx",
            "qc_flag", "q_score", "kept_modalities",
            "dist_to_onset_s", "onset_idx_nearest", "onset_time_nearest_s",
            "w_label_mean", "active_prob_mean", "is_transition",
        ]
        for mk in meta_keys:
            if mk in z:
                val = np.asarray(z[mk])[lid]
                if isinstance(val, np.generic):
                    val = val.item()
                if isinstance(val, (bytes, bytearray)):
                    val = val.decode("utf-8", errors="ignore")
                out[mk] = val

        if "task_code" not in out and "task" in out:
            out["task_code"] = out["task"]
        if "trial_id" not in out and "trial" in out:
            out["trial_id"] = out["trial"]

        # ---------------------------
        # Commit supervision (FIXED)
        #   - NEVER supervise REST windows (commit_mask stays 0)
        #   - Only supervise ACTION near onset (small window around onset)
        # ---------------------------
        READY_HORIZON_S = 2.00   # y_ready=1 when 0..READY_HORIZON_S after onset
        SUP_PRE_S      = 0.50    # include small pre-onset supervised NEG region
        SUP_POST_S     = 2.00    # supervise only up to +SUP_POST_S around onset
        READY_MIN_PROB = 0.80

        y_ready = 0.0
        commit_mask = 0.0  # 1.0 onset-based, 0.5 proxy-based, 0.0 none

        # ✅ IMPORTANT: do NOT supervise commit during REST
        if ya_int == 1:
            is_tr = int(out.get("is_transition", 0))

            if is_tr == 0:
                # 1) Onset-based (strong) — only near onset window
                if ("dist_to_onset_s" in out) and ("onset_idx_nearest" in out):
                    try:
                        d  = float(out["dist_to_onset_s"])
                        oi = int(out["onset_idx_nearest"])
                        if np.isfinite(d) and (oi >= 0) and (-SUP_PRE_S <= d <= SUP_POST_S):
                            commit_mask = 1.0
                            y_ready = 1.0 if (0.0 <= d <= READY_HORIZON_S) else 0.0
                    except Exception:
                        pass

                # 2) Proxy-based (weaker) — only if not already onset-supervised
                if commit_mask == 0.0:
                    try:
                        if "active_prob_mean" in out:
                            ap = float(out["active_prob_mean"])
                            if np.isfinite(ap):
                                commit_mask = 0.5
                                y_ready = 1.0 if (ap >= READY_MIN_PROB) else 0.0
                        elif "w_label_mean" in out:
                            wl = float(out["w_label_mean"])
                            if np.isfinite(wl):
                                commit_mask = 0.5
                                y_ready = 1.0 if (wl >= READY_MIN_PROB) else 0.0
                    except Exception:
                        pass

        out["y_ready"] = torch.tensor(float(y_ready), dtype=torch.float32)
        out["commit_mask"] = torch.tensor(float(commit_mask), dtype=torch.float32)

        return out



class ShardAwareActionBalancedBatchSampler(torch.utils.data.Sampler[List[int]]):
    def __init__(self, ds: ShardNPZDataset, batch_size: int, seed: int = 42):
        self.ds = ds
        self.bs = int(batch_size)
        self.seed = int(seed)
        self.epoch = 0

        y = ds.action_labels_subset()
        if y is None:
            raise RuntimeError("Dataset has no action index; set build_action_index=True.")

        by_shard = defaultdict(list)
        for i in range(len(ds)):
            by_shard[int(ds.sid_of_i[i])].append(i)

        self.shards = sorted(by_shard.keys())
        self.idxs_by_shard = {sid: np.asarray(by_shard[sid], dtype=np.int64) for sid in self.shards}

        y_sub = np.asarray(y, dtype=np.int64)
        c0 = max(1, int((y_sub == 0).sum()))
        c1 = max(1, int((y_sub == 1).sum()))
        w0 = 1.0 / c0
        w1 = 1.0 / c1
        self.weights = np.where(y_sub == 0, w0, w1).astype(np.float64)

        shard_mass = []
        self.w_by_shard = {}
        for sid in self.shards:
            idxs = self.idxs_by_shard[sid]
            w = self.weights[idxs]
            self.w_by_shard[sid] = w
            shard_mass.append(w.sum())
        shard_mass = np.asarray(shard_mass, dtype=np.float64)
        self.p_shard = shard_mass / max(1e-12, shard_mass.sum())

        self.num_samples = len(ds)

    def set_epoch(self, epoch: int):
        self.epoch = int(epoch)

    def __len__(self):
        return int(math.ceil(self.num_samples / self.bs))

    def __iter__(self):
        rng = np.random.RandomState(self.seed + 9973 * self.epoch)
        produced = 0
        while produced < self.num_samples:
            this_bs = min(self.bs, self.num_samples - produced)
            sid = int(rng.choice(self.shards, p=self.p_shard))
            idxs = self.idxs_by_shard[sid]
            w = self.w_by_shard[sid]
            pw = w / max(1e-12, w.sum())
            chosen = rng.choice(idxs, size=this_bs, replace=True, p=pw)
            produced += this_bs
            yield chosen.tolist()

# ============================================================
# DataLoader builder
# ============================================================
def _mp_ctx_fork_or_none():
    if os.name != "posix":
        return None
    try:
        return tmp.get_context("fork")
    except Exception:
        return None

def make_loaders(
    dataset_dir: Path,
    exp_prefix: str,
    fold: int,
    bs_train: int,
    bs_eval: int,
    num_workers: int,
    label_frac: float,
    seed: int,
    use_p55_features: Any,
    feature_pattern: str,
    strict_feature_align: bool,
    use_action_sampler: bool,
    sampler_mode: str,
    rel_dim: Optional[int],
    cache_size: int,
    device: str,
):
    fold_dir = dataset_dir / f"{exp_prefix}_fold{fold}"
    if not fold_dir.exists():
        raise FileNotFoundError(f"Missing fold dir: {fold_dir}")

    feat_train = feat_val = feat_test = None
    use_feats_final = False
    if use_p55_features in (True, "auto"):
        feat_train = find_feature_file(dataset_dir, fold, "train", feature_pattern)
        feat_val   = find_feature_file(dataset_dir, fold, "val",   feature_pattern)
        feat_test  = find_feature_file(dataset_dir, fold, "test",  feature_pattern)
        if feat_train and feat_val and feat_test:
            use_feats_final = True
        elif use_p55_features is True:
            raise FileNotFoundError(
                f"use_p55_features=True but missing feature files.\ntrain={feat_train}\nval={feat_val}\ntest={feat_test}"
            )
    print(f"[feat] fold{fold} train={feat_train} val={feat_val} test={feat_test} -> use_feats={use_feats_final}")


    # rel_dim infer if None
    DEFAULT_REL_DIM = 4
    MAX_REASONABLE_REL_DIM = 64
    rel_dim_guess = int(rel_dim) if rel_dim is not None else DEFAULT_REL_DIM

    if rel_dim is None:
        try:
            first_shard = next(iter(sorted((fold_dir / "train").glob("*_shard_*.npz"))), None)
            if first_shard is not None:
                with np.load(first_shard, allow_pickle=False) as z:
                    if "r_eeg" in z.files:
                        r = np.asarray(z["r_eeg"])
                        d = int(r.shape[-1]) if r.ndim >= 1 else DEFAULT_REL_DIM
                        rel_dim_guess = d if (1 <= d <= MAX_REASONABLE_REL_DIM) else DEFAULT_REL_DIM
        except Exception:
            rel_dim_guess = DEFAULT_REL_DIM

    ds_tr = ShardNPZDataset(
        fold_dir / "train",
        keep_frac=float(label_frac),
        seed=seed,
        feature_file=feat_train if use_feats_final else None,
        strict_feature_align=bool(strict_feature_align),
        build_action_index=bool(use_action_sampler),
        rel_dim=int(rel_dim_guess),
        cache_size=int(cache_size),
    )
    ds_va = ShardNPZDataset(
        fold_dir / "val",
        keep_frac=1.0,
        seed=seed,
        feature_file=feat_val if use_feats_final else None,
        strict_feature_align=bool(strict_feature_align),
        build_action_index=False,
        rel_dim=int(rel_dim_guess),
        cache_size=int(cache_size),
    )
    ds_te = ShardNPZDataset(
        fold_dir / "test",
        keep_frac=1.0,
        seed=seed,
        feature_file=feat_test if use_feats_final else None,
        strict_feature_align=bool(strict_feature_align),
        build_action_index=False,
        rel_dim=int(rel_dim_guess),
        cache_size=int(cache_size),
    )

    s0 = ds_tr[0]
    Ce = int(s0["X_EEG"].shape[-1])
    Cm = int(s0["X_EMG"].shape[-1])
    Ct = int(s0["X_ET"].shape[-1])

    r0 = s0["r_eeg"]
    rel_dim_infer = int(r0.shape[-1]) if (torch.is_tensor(r0) and r0.ndim >= 1) else int(np.asarray(r0).shape[-1])

    is_cuda = str(device).startswith("cuda")
    pin = bool(is_cuda)

    mp_ctx = _mp_ctx_fork_or_none()
    nw_tr = int(min(num_workers, os.cpu_count() or 4))
    nw_ev = int(max(0, nw_tr // 2))

    common_tr = dict(num_workers=nw_tr, pin_memory=pin)
    if nw_tr > 0:
        common_tr["persistent_workers"] = True
        common_tr["prefetch_factor"] = 4
        if mp_ctx is not None:
            common_tr["multiprocessing_context"] = mp_ctx

    common_ev = dict(num_workers=nw_ev, pin_memory=pin)
    if nw_ev > 0:
        common_ev["persistent_workers"] = True
        common_ev["prefetch_factor"] = 2
        if mp_ctx is not None:
            common_ev["multiprocessing_context"] = mp_ctx

    sampler_enabled = False
    batch_sampler = None

    if use_action_sampler and sampler_mode == "shard_aware":
        batch_sampler = ShardAwareActionBalancedBatchSampler(ds_tr, batch_size=int(bs_train), seed=seed)
        sampler_enabled = True
        tr_loader = torch.utils.data.DataLoader(ds_tr, batch_sampler=batch_sampler, **common_tr)
        y = ds_tr.action_labels_subset()
        c0 = int((y == 0).sum()); c1 = int((y == 1).sum())
        print(f"[sampler shard-aware] train c0={c0}, c1={c1}, ratio={c1/max(1,c0):.3f}")
    else:
        tr_loader = torch.utils.data.DataLoader(
            ds_tr, batch_size=int(bs_train), shuffle=True, drop_last=True, **common_tr
        )

    va_loader = torch.utils.data.DataLoader(
        ds_va, batch_size=int(bs_eval), shuffle=False, drop_last=False, **common_ev
    )
    te_loader = torch.utils.data.DataLoader(
        ds_te, batch_size=int(bs_eval), shuffle=False, drop_last=False, **common_ev
    )

    return tr_loader, va_loader, te_loader, Ce, Cm, Ct, 0, rel_dim_infer, use_feats_final, sampler_enabled, batch_sampler

# ============================================================
# Train augmentations
# ============================================================
def apply_train_augs(
    batch,
    p_time_shift=0.10, max_shift=6,
    p_time_mask=0.08, mask_frac=0.06,
    p_chan_drop=0.06, chan_drop_frac=0.06,
    p_mod_drop=0.03,
):
    def time_shift(X, M):
        if random.random() > p_time_shift:
            return X, M
        s = random.randint(-max_shift, max_shift)
        if s == 0:
            return X, M
        return torch.roll(X, shifts=s, dims=1), torch.roll(M, shifts=s, dims=1)

    def time_mask(X, M):
        if random.random() > p_time_mask:
            return X, M
        B, T, C = X.shape
        w = max(1, int(T * mask_frac))
        t0 = random.randint(0, max(0, T - w))
        X[:, t0:t0+w, :] = 0.0
        M[:, t0:t0+w, :] = 0.0
        return X, M

    def chan_drop(X, M):
        if random.random() > p_chan_drop:
            return X, M
        B, T, C = X.shape
        k = max(1, int(C * chan_drop_frac))
        drop = torch.randperm(C, device=X.device)[:k]
        X[:, :, drop] = 0.0
        M[:, :, drop] = 0.0
        return X, M

    def modality_drop(Xe, Me, Xm, Mm, Xt, Mt, re, rm, rt):
        if random.random() > p_mod_drop:
            return Xe, Me, Xm, Mm, Xt, Mt, re, rm, rt
        m = random.choice([0, 1, 2])
        if m == 0:
            Xe.zero_(); Me.zero_(); re.mul_(0.0)
            if "F_psd" in batch: batch["F_psd"].zero_()
        elif m == 1:
            Xm.zero_(); Mm.zero_(); rm.mul_(0.0)
            if "F_emg" in batch: batch["F_emg"].zero_()
        else:
            Xt.zero_(); Mt.zero_(); rt.mul_(0.0)
        if "F_mask" in batch:
            if m == 0: batch["F_mask"][:, 0] = 0.0
            if m == 1: batch["F_mask"][:, 1] = 0.0
            if m == 2: batch["F_mask"][:, 2] = 0.0
        return Xe, Me, Xm, Mm, Xt, Mt, re, rm, rt

    Xe, Me = batch["X_EEG"], batch["M_EEG"]
    Xm, Mm = batch["X_EMG"], batch["M_EMG"]
    Xt, Mt = batch["X_ET"],  batch["M_ET"]

    Xe, Me = time_shift(Xe, Me); Xm, Mm = time_shift(Xm, Mm); Xt, Mt = time_shift(Xt, Mt)
    Xe, Me = time_mask(Xe, Me);  Xm, Mm = time_mask(Xm, Mm);  Xt, Mt = time_mask(Xt, Mt)
    Xe, Me = chan_drop(Xe, Me);  Xm, Mm = chan_drop(Xm, Mm);  Xt, Mt = chan_drop(Xt, Mt)

    re = batch["r_eeg"]; rm = batch["r_emg"]; rt = batch["r_et"]
    Xe, Me, Xm, Mm, Xt, Mt, re, rm, rt = modality_drop(Xe, Me, Xm, Mm, Xt, Mt, re, rm, rt)

    batch["X_EEG"], batch["M_EEG"] = Xe, Me
    batch["X_EMG"], batch["M_EMG"] = Xm, Mm
    batch["X_ET"],  batch["M_ET"]  = Xt, Mt
    batch["r_eeg"], batch["r_emg"], batch["r_et"] = re, rm, rt
    return batch


# ============================================================
# Debug helper (put once)
# ============================================================
@torch.no_grad()
def debug_batch(batch):
    ae = (batch["M_EEG"].float().mean(dim=(1,2)) > 0.05).float()
    am = (batch["M_EMG"].float().mean(dim=(1,2)) > 0.05).float()
    at = (batch["M_ET" ].float().mean(dim=(1,2)) > 0.05).float()
    avail = torch.stack([ae,am,at], dim=1)
    mask2 = (avail.sum(dim=1) >= 2).float()

    print("avail sums:", avail.sum(dim=1).detach().cpu().numpy()[:20])
    print("mask2_mean:", float(mask2.mean().item()))
    print("all_zero_frac:", float((avail.sum(dim=1)==0).float().mean().item()))

    if "commit_mask" in batch:
        cm = batch["commit_mask"].float()
        print("commit_mask mean:", float(cm.mean().item()),
              " frac>0:", float((cm>0).float().mean().item()),
              " frac==1:", float((cm==1).float().mean().item()))

    # ✅ ADD THIS BLOCK HERE (same indent as the if above)
    if ("y_ready" in batch) and ("commit_mask" in batch):
        yr = batch["y_ready"].float()
        cm = batch["commit_mask"].float()
        m = (cm > 0)
        pos_rate = float(((yr[m] > 0.5).float().mean().item())) if m.any() else float("nan")
        print("y_ready pos rate (cm>0):", pos_rate)

        
        
# ============================================================
# Model helpers
# ============================================================
def safe_norm(x: torch.Tensor, dim: int = -1, eps: float = 1e-12) -> torch.Tensor:
    return x / (x.norm(dim=dim, keepdim=True).clamp_min(eps))

def patchify_time_mask(M: torch.Tensor, patch: int, thr: float = 0.10) -> torch.Tensor:
    B, T, C = M.shape
    T2 = (T // patch) * patch
    M = M[:, :T2, :]
    m_time = (M.mean(dim=-1) > 0).float()
    L = T2 // patch
    m_patch = m_time.reshape(B, L, patch).mean(dim=-1)
    return (m_patch > thr).float()

# ============================================================
# IROS-ready interface objects
# ============================================================
@dataclass
class IntentPacket:
    readiness_ct: float
    discrete_state: str
    action_prob: float
    task_probs: List[float]
    w_eeg: float
    w_emg: float
    w_et: float
    avail_eeg: float
    avail_emg: float
    avail_et: float
    stability_var: float

    def to_json(self) -> str:
        return json.dumps(asdict(self))

class CommitDecisionFilter:
    def __init__(
        self,
        thr_on: float = 0.80,
        thr_off: float = 0.55,
        dwell_windows: int = 2,
        cooldown_windows: int = 0,
    ):
        self.thr_on = float(thr_on)
        self.thr_off = float(thr_off)
        self.dwell = int(max(1, dwell_windows))
        self.cooldown = int(max(0, cooldown_windows))

        self.state = "HOLD"
        self._on_count = 0
        self._cooldown_left = 0

    def reset(self):
        self.state = "HOLD"
        self._on_count = 0
        self._cooldown_left = 0

    def step(self, ct: float) -> Tuple[str, bool]:
        ct = float(ct)
        commit_event = False

        if self._cooldown_left > 0:
            self._cooldown_left -= 1

        if self.state == "HOLD":
            if self._cooldown_left > 0:
                self._on_count = 0
                return self.state, False

            if ct >= self.thr_on:
                self._on_count += 1
            else:
                self._on_count = 0

            if self._on_count >= self.dwell:
                self.state = "COMMIT"
                commit_event = True
                self._on_count = 0
            return self.state, commit_event

        if ct <= self.thr_off:
            self.state = "HOLD"
            if self.cooldown > 0:
                self._cooldown_left = self.cooldown
        return self.state, False

# ============================================================
# Configs
# ============================================================
@dataclass
class NeuroCommitCfg:
    d_model: int = 160
    drop: float = 0.10
    patch: int = 25
    rel_dim: int = 4

    eeg_virtual_K: int = 6
    eeg_sinc_filters: int = 6
    eeg_graph_heads: int = 4

    emg_synergy_M: int = 4
    emg_env_kernel: int = 25
    emg_burst_kernel: int = 9

    et_event_bins: int = 4
    et_event_topk: int = 3

    fuse_use_uncertainty: bool = True
    fuse_ssm_hidden: int = 192
    fuse_ssm_layers: int = 1

    # Commit decision filter params (robotics-facing)
    commit_thr_on: float = 0.80
    commit_thr_off: float = 0.55
    commit_dwell_windows: int = 2
    commit_cooldown_windows: int = 0

    # Commit head network
    commit_hidden: int = 160
    commit_attn_pool: bool = True

    proj_dim: int = 160

    # ✅ Phase 5.5 feature usage (fixed dims for v6_3)
    use_p55_features: bool = True
    feat_dim_psd: int = 96
    feat_dim_emg: int = 24
    feat_dim_mask: int = 3
    feat_z_scale: float = 0.25
    feat_token_scale: float = 0.25

# ============================================================
# Patch projection
# ============================================================
class PatchProject1D(nn.Module):
    def __init__(self, Cin: int, D: int, patch: int, drop: float):
        super().__init__()
        self.patch = int(patch)
        self.proj = nn.Conv1d(Cin, D, kernel_size=self.patch, stride=self.patch, bias=False)
        self.norm = nn.LayerNorm(D)
        self.drop = nn.Dropout(drop)

    def forward(self, X: torch.Tensor, M: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, C = X.shape
        p = self.patch
        T2 = (T // p) * p
        if T2 <= 0:
            raise RuntimeError("T too small for patch size")
        X = X[:, :T2, :]
        M = M[:, :T2, :]

        x = (X * M).transpose(1, 2)
        h = self.proj(self.drop(x)).transpose(1, 2)     # (B,L,D)
        h = self.norm(h)

        m_patch = patchify_time_mask(M, patch=p, thr=0.10)  # (B,L)
        return h, m_patch

# ============================================================
# EEG encoder
# ============================================================
class DilatedDWGLUBlock(nn.Module):
    def __init__(self, C: int, k: int = 9, d: int = 1, drop: float = 0.1):
        super().__init__()
        pad = (k // 2) * d
        self.dw = nn.Conv1d(C, C, kernel_size=k, padding=pad, dilation=d, groups=C, bias=False)
        self.pw = nn.Conv1d(C, 2 * C, kernel_size=1, bias=False)
        self.norm = nn.GroupNorm(num_groups=max(1, C), num_channels=C)
        self.drop = nn.Dropout(drop)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        y = self.dw(x)
        y = self.pw(y)
        a, b = y.chunk(2, dim=1)
        y = F.gelu(a) * torch.sigmoid(b)
        y = self.drop(y)
        return self.norm(x + y)

class TokenGraphMixer(nn.Module):
    def __init__(self, K: int, drop: float = 0.1):
        super().__init__()
        self.q = nn.Linear(K, K, bias=False)
        self.k = nn.Linear(K, K, bias=False)
        self.proj = nn.Linear(K, K, bias=False)
        self.drop = nn.Dropout(drop)
        self.norm = nn.LayerNorm(K)

    def forward(self, Xv: torch.Tensor, Mv: torch.Tensor) -> torch.Tensor:
        mp = (Mv > 0).float()
        den = mp.sum(dim=1).clamp_min(1.0)
        s = (Xv * mp).sum(dim=1) / den

        q = self.q(s)
        k = self.k(s)
        A = torch.softmax((q.unsqueeze(-1) @ k.unsqueeze(-2)) / math.sqrt(Xv.shape[-1]), dim=-1)

        Y = torch.einsum("bti,bij->btj", Xv, A)
        Y = self.drop(self.proj(Y))
        return self.norm(Xv + Y)

def _masked_channel_standardize(X: torch.Tensor, M: torch.Tensor, eps: float = 1e-5):
    m = (M > 0).float()
    den = m.sum(dim=1, keepdim=True).clamp_min(1.0)
    mu = (X * m).sum(dim=1, keepdim=True) / den
    var = ((X - mu) ** 2 * m).sum(dim=1, keepdim=True) / den
    Xz = (X - mu) / torch.sqrt(var + eps)
    return Xz * m

class EEG_VEMGraphBank(nn.Module):
    def __init__(self, Ce: int, cfg: NeuroCommitCfg):
        super().__init__()
        assert Ce == 8, "EEG encoder expects 8ch EEG"

        self.Ce = int(Ce)
        self.K  = int(cfg.eeg_virtual_K)
        self.Fb = int(cfg.eeg_sinc_filters)
        self.D  = int(cfg.d_model)
        self.drop = float(cfg.drop)

        ks_all = [9, 17, 33, 65, 129, 161]
        ks = ks_all[:max(1, min(self.Fb, len(ks_all)))]
        self.ms = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(self.Ce, self.Ce, kernel_size=k, padding=k//2, groups=self.Ce, bias=False),
                nn.GELU()
            )
            for k in ks
        ])
        self.ms_mix = nn.Conv1d(self.Ce * len(ks), self.Ce, kernel_size=1, bias=False)
        self.ms_norm = nn.GroupNorm(num_groups=self.Ce, num_channels=self.Ce)

        self.low = nn.Conv1d(self.Ce, self.Ce, kernel_size=151, padding=75, groups=self.Ce, bias=False)

        desc_dim = 6
        self.token_mlp = nn.Sequential(
            nn.Linear(desc_dim, 64),
            nn.GELU(),
            nn.Dropout(self.drop),
            nn.Linear(64, self.K),
        )

        self.token_graph = TokenGraphMixer(self.K, drop=self.drop)

        n_layers = int(max(1, getattr(cfg, "eeg_graph_heads", 3)))
        dilations = [1, 2, 4, 8, 1, 2][:n_layers]
        self.temporal = nn.Sequential(*[
            DilatedDWGLUBlock(C=self.K, k=9, d=int(d), drop=self.drop)
            for d in dilations
        ])

        self.patch = PatchProject1D(Cin=self.K, D=self.D, patch=cfg.patch, drop=self.drop)

    def forward(self, X: torch.Tensor, M: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, Ce = X.shape
        assert Ce == self.Ce

        Xz = _masked_channel_standardize(X, M)

        x = Xz.transpose(1, 2)
        feats = [f(x) for f in self.ms]
        fb = torch.cat(feats, dim=1)
        xf = self.ms_norm(self.ms_mix(fb))
        xf_t = xf.transpose(1, 2)

        avail = M.mean(dim=1).clamp(0, 1)

        m = (M > 0).float()
        den = m.sum(dim=1).clamp_min(1.0)
        mu = (xf_t * m).sum(dim=1) / den
        var = (((xf_t - mu.unsqueeze(1)) ** 2) * m).sum(dim=1) / den
        std = torch.sqrt(var + 1e-6)
        energy = ((xf_t ** 2) * m).sum(dim=1) / den

        low = self.low(xf).transpose(1, 2)
        d1 = (low[:, 1:, :] - low[:, :-1, :]).abs()
        absdiff = (d1 * m[:, 1:, :]).sum(dim=1) / (m[:, 1:, :].sum(dim=1).clamp_min(1.0))
        slope = absdiff

        desc = torch.stack([mu, std, energy, slope, absdiff, avail], dim=-1)

        logits = self.token_mlp(desc)
        logits = logits.permute(0, 2, 1).contiguous()
        logits = logits + torch.log(avail.unsqueeze(1).clamp_min(1e-6))
        W = torch.softmax(logits, dim=-1)

        Xv = torch.einsum("btc,bkc->btk", xf_t, W)
        Mv = torch.einsum("btc,bkc->btk", M, W).clamp(0, 1)

        Xv = self.token_graph(Xv, Mv)
        Xv_c = Xv.transpose(1, 2)
        Xv_c = self.temporal(Xv_c)
        Xv = Xv_c.transpose(1, 2)

        H, m_patch = self.patch(Xv, Mv)
        return H, m_patch

# ============================================================
# EMG encoder
# ============================================================
class EMG_SyEMB(nn.Module):
    def __init__(self, Cm: int, cfg: NeuroCommitCfg):
        super().__init__()
        assert Cm == 4, "EMG_SyEMB expects 4ch EMG"
        self.Msyn = int(cfg.emg_synergy_M)
        self.D = int(cfg.d_model)
        self.drop = float(cfg.drop)

        k_env = int(cfg.emg_env_kernel)
        k_bur = int(cfg.emg_burst_kernel)

        self.smooth_env = nn.Conv1d(Cm, Cm, kernel_size=k_env, padding=k_env // 2, groups=Cm, bias=False)
        self.smooth_burst = nn.Conv1d(Cm, Cm, kernel_size=k_bur, padding=k_bur // 2, groups=Cm, bias=False)

        self.synergy_raw = nn.Parameter(torch.zeros(self.Msyn, Cm))

        self.pre = nn.Sequential(
            nn.Conv1d(self.Msyn, self.Msyn, kernel_size=9, padding=4, groups=self.Msyn, bias=False),
            nn.GELU(),
            nn.Conv1d(self.Msyn, self.Msyn, kernel_size=1, bias=False),
            nn.GELU(),
            nn.Dropout(self.drop),
        )

        self.patch = PatchProject1D(Cin=self.Msyn, D=self.D, patch=cfg.patch, drop=self.drop)

    def forward(self, X: torch.Tensor, M: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        x = (X * M).transpose(1, 2)

        env = self.smooth_env(torch.abs(x))
        dx = torch.zeros_like(x)
        dx[:, :, 1:] = x[:, :, 1:] - x[:, :, :-1]
        burst = self.smooth_burst(torch.abs(dx))

        xm = 0.9 * env + 0.6 * burst + 0.2 * x

        W = F.softplus(self.synergy_raw)
        W = W / (W.sum(dim=1, keepdim=True).clamp_min(1e-6))
        S = torch.einsum("mc,bct->bmt", W, xm)

        Mm = (M.transpose(1, 2) > 0).float()
        Sm = torch.einsum("mc,bct->bmt", W, Mm).clamp(0, 1)

        S = self.pre(S).transpose(1, 2)
        Sm = Sm.transpose(1, 2)

        H, m_patch = self.patch(S, Sm)
        return H, m_patch

# ============================================================
# ET encoder (RAVEN-ET)
# ============================================================
class MaskedPatchify1D(nn.Module):
    def __init__(self, D: int, patch: int):
        super().__init__()
        self.patch = int(patch)
        self.conv = nn.Conv1d(D, D, kernel_size=self.patch, stride=self.patch, bias=False)
        self.ln = nn.LayerNorm(D)

    def forward(self, Hd: torch.Tensor, m_time: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, D = Hd.shape
        p = self.patch
        T2 = (T // p) * p
        Hd = Hd[:, :T2, :]
        m_time = m_time[:, :T2]

        x = Hd.transpose(1, 2)
        Hp = self.conv(x).transpose(1, 2)

        mv = F.avg_pool1d(m_time.unsqueeze(1), kernel_size=p, stride=p).squeeze(1)
        m_patch = (mv > 0.10).float()

        Hp = Hp / mv.clamp_min(0.10).unsqueeze(-1)
        Hp = self.ln(Hp)
        return Hp, m_patch

class VBTransformerBlock(nn.Module):
    def __init__(self, D: int, nhead: int = 4, drop: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=D, num_heads=nhead, dropout=drop, batch_first=True)
        self.ln1 = nn.LayerNorm(D)
        self.ff = nn.Sequential(
            nn.Linear(D, 4 * D),
            nn.GELU(),
            nn.Dropout(drop),
            nn.Linear(4 * D, D),
            nn.Dropout(drop),
        )
        self.ln2 = nn.LayerNorm(D)

    def forward(self, x: torch.Tensor, key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        kpm = key_padding_mask
        if kpm is not None:
            kpm = kpm.to(torch.bool)
            if kpm.ndim == 2:
                all_mask = kpm.all(dim=1)
                if all_mask.any():
                    kpm = kpm.clone()
                    kpm[all_mask, 0] = False

        if x.is_cuda:
            with torch.autocast(device_type="cuda", enabled=False):
                y, _ = self.attn(x.float(), x.float(), x.float(), key_padding_mask=kpm, need_weights=False)
            y = y.to(x.dtype)
        else:
            y, _ = self.attn(x, x, x, key_padding_mask=kpm, need_weights=False)

        x = self.ln1(x + y)
        x = self.ln2(x + self.ff(x))
        return x

class ET_ECPT(nn.Module):
    def __init__(self, Ct: int, cfg: NeuroCommitCfg):
        super().__init__()
        self.Ct = int(Ct)
        self.D = int(cfg.d_model)
        self.drop = float(cfg.drop)
        self.patch = int(cfg.patch)

        self.topk = int(cfg.et_event_topk)
        self.event_bins = int(cfg.et_event_bins)
        self.nhead = 4

        self.dense = nn.Sequential(
            nn.Linear(self.Ct + 3, self.D),
            nn.GELU(),
            nn.Dropout(self.drop),
            nn.Linear(self.D, self.D),
            nn.GELU(),
            nn.Dropout(self.drop),
        )

        self.patchify = MaskedPatchify1D(D=self.D, patch=self.patch)

        self.ev_emb = nn.Embedding(self.event_bins * 2, self.D)
        self.ev_mag = nn.Sequential(nn.Linear(1, self.D), nn.Tanh())
        self.ev_ln = nn.LayerNorm(self.D)

        self.cross = nn.MultiheadAttention(embed_dim=self.D, num_heads=self.nhead, dropout=self.drop, batch_first=True)
        self.cross_ln = nn.LayerNorm(self.D)

        self.self1 = VBTransformerBlock(D=self.D, nhead=self.nhead, drop=self.drop)

        self.gate = nn.Sequential(
            nn.Linear(self.D * 2, self.D),
            nn.GELU(),
            nn.Dropout(self.drop),
            nn.Linear(self.D, self.D),
            nn.Sigmoid(),
        )

    def forward(self, X: torch.Tensor, M: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        B, T, C = X.shape
        p = self.patch
        T2 = (T // p) * p
        if T2 <= 0:
            return X.new_zeros((B, 0, self.D)), X.new_zeros((B, 0))

        X = X[:, :T2, :]
        M = M[:, :T2, :]

        v = M.mean(dim=-1, keepdim=True).clamp(0, 1)
        m_time = (v.squeeze(-1) > 0.05).float()

        dx = torch.zeros_like(X)
        dx[:, 1:, :] = X[:, 1:, :] - X[:, :-1, :]
        dd = torch.zeros_like(X)
        dd[:, 2:, :] = dx[:, 2:, :] - dx[:, 1:-1, :]

        vel = torch.sqrt((dx * dx).mean(dim=-1, keepdim=True).clamp_min(1e-12))
        acc = torch.sqrt((dd * dd).mean(dim=-1, keepdim=True).clamp_min(1e-12))

        Xd = torch.cat([X, vel, acc, v], dim=-1)
        Hd = self.dense(Xd) * v

        Hp, m_patch = self.patchify(Hd, m_time)
        patch_kpm = (m_patch <= 0.0)

        e = (0.7 * vel + 0.3 * acc) * v
        e_flat = e.squeeze(-1)
        e_flat = e_flat.masked_fill(m_time <= 0.0, -1e9)

        k = min(self.topk, T2)
        if k <= 0:
            H = self.self1(Hp, key_padding_mask=patch_kpm)
            H = H * m_patch.unsqueeze(-1)
            return H, m_patch

        vals, idx = torch.topk(e_flat, k=k, dim=1, largest=True)
        vals = vals.clamp_min(0.0)

        bin_w = max(1, T2 // self.event_bins)
        bin_id = torch.clamp(idx // bin_w, min=0, max=self.event_bins - 1)

        dx_mean = dx.mean(dim=-1)
        sign = (dx_mean.gather(1, idx) >= 0).long()
        ev_id = (bin_id * 2 + sign).long()

        ev = self.ev_emb(ev_id) + self.ev_mag(vals.unsqueeze(-1))
        ev = self.ev_ln(ev)
        ev = ev * (vals > 0).float().unsqueeze(-1)

        if Hp.is_cuda:
            with sdp_math_only():
                with torch.autocast(device_type="cuda", enabled=False):
                    inj, _ = self.cross(query=Hp.float(), key=ev.float(), value=ev.float(), need_weights=False)
            inj = inj.to(Hp.dtype)
        else:
            inj, _ = self.cross(query=Hp, key=ev, value=ev, need_weights=False)

        Hp2 = self.cross_ln(Hp + inj)

        g = self.gate(torch.cat([Hp2, inj], dim=-1))
        H = Hp2 + g * inj

        H = self.self1(H, key_padding_mask=patch_kpm)
        H = H * m_patch.unsqueeze(-1)
        return H, m_patch

# ============================================================
# Fusion (sequence-preserving) + Commit head (temporal)
# ============================================================
class UncertaintyFusionSSM(nn.Module):
    def __init__(self, cfg: NeuroCommitCfg):
        super().__init__()
        D = int(cfg.d_model)
        self.rel_dim = int(cfg.rel_dim)
        self.use_unc = bool(cfg.fuse_use_uncertainty)

        self.u_mlp = nn.Sequential(
            nn.Linear(self.rel_dim + 1, D),
            nn.GELU(),
            nn.Dropout(float(cfg.drop)),
            nn.Linear(D, 1),
        )

        self.gru = nn.GRU(
            input_size=D,
            hidden_size=int(cfg.fuse_ssm_hidden),
            num_layers=int(cfg.fuse_ssm_layers),
            batch_first=True,
            bidirectional=False
        )
        self.out = nn.Sequential(
            nn.LayerNorm(int(cfg.fuse_ssm_hidden)),
            nn.Linear(int(cfg.fuse_ssm_hidden), D),
            nn.GELU(),
            nn.Dropout(float(cfg.drop)),
            nn.Linear(D, D),
        )

    def forward(
        self,
        He: torch.Tensor, Hm: torch.Tensor, Ht: torch.Tensor,
        me: torch.Tensor, mm: torch.Tensor, mt: torch.Tensor,
        r_eeg: torch.Tensor, r_emg: torch.Tensor, r_et: torch.Tensor,
    ):
        B, L, D = He.shape

        ae = me.mean(dim=1, keepdim=True)
        am = mm.mean(dim=1, keepdim=True)
        at = mt.mean(dim=1, keepdim=True)

        if self.use_unc:
            ue = self.u_mlp(torch.cat([r_eeg, ae], dim=1))
            um = self.u_mlp(torch.cat([r_emg, am], dim=1))
            ut = self.u_mlp(torch.cat([r_et,  at], dim=1))
            U = torch.cat([ue, um, ut], dim=1)
            A = torch.cat([ae, am, at], dim=1).clamp(0, 1)
            A = (A > 0.05).float()
            logits = -U + torch.log(A.clamp_min(1e-6))
            w = torch.softmax(logits, dim=1)
        else:
            A = torch.cat([(ae > 0.05).float(), (am > 0.05).float(), (at > 0.05).float()], dim=1)
            w = A / (A.sum(dim=1, keepdim=True).clamp_min(1e-12))

        w_e = w[:, 0].view(B, 1, 1)
        w_m = w[:, 1].view(B, 1, 1)
        w_t = w[:, 2].view(B, 1, 1)

        Hf = w_e * He + w_m * Hm + w_t * Ht

        out_seq, hN = self.gru(Hf)
        h = hN[-1]
        z = self.out(h)

        st = {
            "w_eeg": w[:, 0],
            "w_emg": w[:, 1],
            "w_et":  w[:, 2],
            "avail_eeg": ae.squeeze(1),
            "avail_emg": am.squeeze(1),
            "avail_et":  at.squeeze(1),
        }
        return z, Hf, st

class CommitDecisionHeadTemporal(nn.Module):
    def __init__(self, cfg: NeuroCommitCfg):
        super().__init__()
        D = int(cfg.d_model)
        H = int(cfg.commit_hidden)
        self.drop = float(cfg.drop)
        self.use_attn_pool = bool(cfg.commit_attn_pool)

        self.gru = nn.GRU(D, H, num_layers=1, batch_first=True, bidirectional=False)

        if self.use_attn_pool:
            self.attn = nn.Sequential(
                nn.Linear(H, H),
                nn.GELU(),
                nn.Dropout(self.drop),
                nn.Linear(H, 1),
            )

        self.mlp = nn.Sequential(
            nn.LayerNorm(H),
            nn.Linear(H, H),
            nn.GELU(),
            nn.Dropout(self.drop),
            nn.Linear(H, 1),
        )
        self.cls = nn.Sequential(
            nn.LayerNorm(H),
            nn.Linear(H, H),
            nn.GELU(),
            nn.Dropout(self.drop),
            nn.Linear(H, 2),
        )

    def forward(self, Hseq: torch.Tensor, m_seq: Optional[torch.Tensor] = None):
        out, h_last = self.gru(Hseq)   # out: (B,T,H)

        if self.use_attn_pool:
            a = self.attn(out).squeeze(-1)     # (B,T)
            a_fp32 = a.float()

            if m_seq is not None:
                # mask=True means "invalid"
                mask = (m_seq <= 0)

                # ✅ if an entire row is masked, unmask token 0 to avoid softmax(all -inf)
                all_masked = mask.all(dim=1)
                if all_masked.any():
                    mask = mask.clone()
                    mask[all_masked, 0] = False

                a_fp32 = a_fp32.masked_fill(mask, torch.finfo(a_fp32.dtype).min)

            w = torch.softmax(a_fp32, dim=1).to(out.dtype).unsqueeze(-1)  # (B,T,1)
            hT = (out * w).sum(dim=1)  # (B,H)
        else:
            hT = h_last[-1]

        ct_logit = self.mlp(hT).squeeze(-1)  # (B,)
        ct = torch.sigmoid(ct_logit)
        logits_state = self.cls(hT)

        stab = out.var(dim=1, unbiased=False).mean(dim=-1)
        d = (out[:, 1:, :] - out[:, :-1, :]).abs().mean(dim=(1, 2)) if out.shape[1] > 1 else out.new_zeros((out.shape[0],))
        aux = {"stability_var": stab, "delta_mean": d}
        return ct, ct_logit, logits_state, aux



# ============================================================
# Gradient checkpointing (tensor-only)
# ============================================================
def _checkpoint_call(fn, *args):
    try:
        return _checkpoint(fn, *args, use_reentrant=False)
    except TypeError:
        return _checkpoint(fn, *args)

class NeuroCommitM3(nn.Module):
    def __init__(self, Ce: int, Cm: int, Ct: int, cfg: NeuroCommitCfg, num_task: int = 5):
        super().__init__()
        self.cfg = cfg
        self.num_task = int(num_task)
        self.gradient_checkpointing = bool(USE_GRADIENT_CHECKPOINTING)

        self.eeg = EEG_VEMGraphBank(Ce, cfg)
        self.emg = EMG_SyEMB(Cm, cfg)
        self.et  = ET_ECPT(Ct, cfg)

        self.fuse = UncertaintyFusionSSM(cfg)

        D = int(cfg.d_model)
        self.head_action = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, D),
            nn.GELU(),
            nn.Dropout(float(cfg.drop)),
            nn.Linear(D, 2),
        )
        self.head_task = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, D),
            nn.GELU(),
            nn.Dropout(float(cfg.drop)),
            nn.Linear(D, self.num_task),
        )

        self.commit = CommitDecisionHeadTemporal(cfg)

        self.proj = nn.Sequential(
            nn.LayerNorm(D),
            nn.Linear(D, int(cfg.proj_dim)),
            nn.GELU(),
            nn.Linear(int(cfg.proj_dim), int(cfg.proj_dim)),
        )
        self.dec_eeg = nn.Sequential(nn.LayerNorm(D), nn.Linear(D, D), nn.GELU(), nn.Linear(D, D))
        self.dec_emg = nn.Sequential(nn.LayerNorm(D), nn.Linear(D, D), nn.GELU(), nn.Linear(D, D))
        self.dec_et  = nn.Sequential(nn.LayerNorm(D), nn.Linear(D, D), nn.GELU(), nn.Linear(D, D))

        # ✅ NEW: Phase 5.5 feature projector (fixed dims)
        feat_dim = int(cfg.feat_dim_psd + cfg.feat_dim_emg + cfg.feat_dim_mask)
        self.feat_dim = feat_dim
        self.use_feats = bool(cfg.use_p55_features)

        self.feat_proj = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Linear(feat_dim, D),
            nn.GELU(),
            nn.Dropout(float(cfg.drop)),
            nn.Linear(D, D),
        )

    def _forward_tensors(self, Xe, Me, Xm, Mm, Xt, Mt, r_eeg, r_emg, r_et, F_cat):
        He, me = self.eeg(Xe, Me)
        Hm, mm = self.emg(Xm, Mm)
        Ht, mt = self.et (Xt, Mt)

        z, Hf, st = self.fuse(He, Hm, Ht, me, mm, mt, r_eeg, r_emg, r_et)

        # ✅ NEW: use Phase 5.5 features
        # F_cat: (B, feat_dim) or (B,0)
        if self.use_feats and (F_cat is not None) and (F_cat.numel() > 0) and (F_cat.shape[1] == self.feat_dim):
            femb = self.feat_proj(F_cat)  # (B,D)
            z = z + float(self.cfg.feat_z_scale) * femb

            # also give commit head a feature token (helps ct under missing sensors)
            ftok = float(self.cfg.feat_token_scale) * femb.unsqueeze(1)  # (B,1,D)
            Hf = torch.cat([Hf, ftok], dim=1)
            # extend modality masks conservatively
            me = torch.cat([me, torch.ones((me.shape[0], 1), device=me.device, dtype=me.dtype)], dim=1)
            mm = torch.cat([mm, torch.ones((mm.shape[0], 1), device=mm.device, dtype=mm.dtype)], dim=1)
            mt = torch.cat([mt, torch.ones((mt.shape[0], 1), device=mt.device, dtype=mt.dtype)], dim=1)

        logits_action = self.head_action(z)
        logits_task   = self.head_task(z)

        m_fuse = ((me + mm + mt) > 0).float().clamp(0, 1)
        ct, ct_logit, logits_state, aux = self.commit(Hf, m_seq=m_fuse)


        return (
            logits_action, logits_task, ct, ct_logit,   # ✅ ADD HERE
            st["w_eeg"], st["w_emg"], st["w_et"],
            st["avail_eeg"], st["avail_emg"], st["avail_et"],
            logits_state, aux["stability_var"], aux["delta_mean"],
            He, Hm, Ht, me, mm, mt, Hf, z
        )


    def forward_window(self, batch: Dict[str, torch.Tensor]):
        Xe, Xm, Xt = batch["X_EEG"], batch["X_EMG"], batch["X_ET"]
        Me, Mm, Mt = batch["M_EEG"], batch["M_EMG"], batch["M_ET"]
        r_eeg = batch["r_eeg"].float()
        r_emg = batch["r_emg"].float()
        r_et  = batch["r_et"].float()

        # ✅ NEW: concatenate features if present
        if self.use_feats and ("F_psd" in batch) and ("F_emg" in batch) and ("F_mask" in batch):
            F_cat = torch.cat([batch["F_psd"], batch["F_emg"], batch["F_mask"]], dim=1).float()
        else:
            # keep tensor-only signature for checkpointing
            F_cat = Xe.new_zeros((Xe.shape[0], 0))

        if self.training and self.gradient_checkpointing:
            outs = _checkpoint_call(self._forward_tensors, Xe, Me, Xm, Mm, Xt, Mt, r_eeg, r_emg, r_et, F_cat)
        else:
            outs = self._forward_tensors(Xe, Me, Xm, Mm, Xt, Mt, r_eeg, r_emg, r_et, F_cat)

        (logits_action, logits_task, ct, ct_logit,
         w_eeg, w_emg, w_et,
         a_eeg, a_emg, a_et,
         commit_logits, stability_var, delta_mean,
         _, _, _, _, _, _, _, _) = outs


        st = {
            "w_eeg": w_eeg, "w_emg": w_emg, "w_et": w_et,
            "avail_eeg": a_eeg, "avail_emg": a_emg, "avail_et": a_et,
            "commit_logits": commit_logits,
            "ct_logit": ct_logit, 
            "stability_var": stability_var,
            "delta_mean": delta_mean,
        }
        return logits_action, logits_task, ct, st

    def forward_ssl(self, batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        Xe, Xm, Xt = batch["X_EEG"], batch["X_EMG"], batch["X_ET"]
        Me, Mm, Mt = batch["M_EEG"], batch["M_EMG"], batch["M_ET"]
        r_eeg = batch["r_eeg"].float()
        r_emg = batch["r_emg"].float()
        r_et  = batch["r_et"].float()

        He, me = self.eeg(Xe, Me)
        Hm, mm = self.emg(Xm, Mm)
        Ht, mt = self.et (Xt, Mt)

        z, Hf, st = self.fuse(He, Hm, Ht, me, mm, mt, r_eeg, r_emg, r_et)

        p = safe_norm(self.proj(z), dim=-1)

        pred_e = self.dec_eeg(Hf)
        pred_m = self.dec_emg(Hf)
        pred_t = self.dec_et (Hf)

        return {
            "proj": p,
            "He": He, "Hm": Hm, "Ht": Ht,
            "me": me, "mm": mm, "mt": mt,
            "pred_e": pred_e, "pred_m": pred_m, "pred_t": pred_t,
            "st": st,
        }

# ============================================================
# SSL helpers
# ============================================================
def _clone_tensor_batch(batch: Dict[str, Any]) -> Dict[str, Any]:
    out = {}
    for k, v in batch.items():
        if torch.is_tensor(v):
            out[k] = v.clone()
        else:
            out[k] = v
    return out

def _mask_patches_inplace(batch: Dict[str, torch.Tensor], patch: int, frac: float):
    if frac <= 0:
        return batch
    for mod, Ckey, Mkey in [("EEG","X_EEG","M_EEG"), ("EMG","X_EMG","M_EMG"), ("ET","X_ET","M_ET")]:
        if Ckey not in batch or Mkey not in batch:
            continue
        X = batch[Ckey]; M = batch[Mkey]
        if not (torch.is_tensor(X) and torch.is_tensor(M)):
            continue
        B, T, C = X.shape
        p = int(patch)
        if p <= 0 or T < p:
            continue
        L = T // p
        if L <= 1:
            continue
        k = max(1, int(round(L * float(frac))))
        for b in range(B):
            idx = torch.randperm(L, device=X.device)[:k]
            for j in idx.tolist():
                t0 = j * p
                t1 = min(T, t0 + p)
                X[b, t0:t1, :].zero_()
                M[b, t0:t1, :].zero_()
        batch[Ckey] = X
        batch[Mkey] = M
    return batch

def _info_nce_symmetric(p1: torch.Tensor, p2: torch.Tensor, temp: float = 0.2) -> torch.Tensor:
    B, D = p1.shape
    logits = (p1 @ p2.t()) / max(1e-6, float(temp))
    labels = torch.arange(B, device=p1.device)
    l12 = F.cross_entropy(logits, labels)
    l21 = F.cross_entropy(logits.t(), labels)
    return 0.5 * (l12 + l21)

def _masked_mse(pred: torch.Tensor, target: torch.Tensor, m_patch: torch.Tensor) -> torch.Tensor:
    mp = m_patch.unsqueeze(-1).float()
    num = ((pred - target) ** 2 * mp).sum()
    den = mp.sum().clamp_min(1.0)
    return num / den

def _apply_ssl_view_augs(batch: Dict[str, torch.Tensor], patch: int, mask_frac: float):
    _mask_patches_inplace(batch, patch=int(patch), frac=float(mask_frac))
    apply_train_augs(
        batch,
        p_time_shift=0.15, max_shift=8,
        p_time_mask=0.12, mask_frac=0.08,
        p_chan_drop=0.10, chan_drop_frac=0.08,
        p_mod_drop=0.15,
    )
    return batch

def train_one_epoch_ssl(
    model: NeuroCommitM3,
    loader,
    opt,
    scaler,
    device: str,
    patch: int,
    temp: float,
    lambda_nce: float,
    lambda_rec: float,
    mask_patch_frac: float,
    use_amp: bool = True,
    grad_clip: float = 1.0,
):
    model.train()
    dev_type = "cuda" if str(device).startswith("cuda") else "cpu"

    total_loss = 0.0
    total_nce = 0.0
    total_rec = 0.0
    n_seen = 0

    for batch in loader:
        batch = to_device(batch, device)

        b1 = _clone_tensor_batch(batch)
        b2 = _clone_tensor_batch(batch)
        _apply_ssl_view_augs(b1, patch=patch, mask_frac=mask_patch_frac)
        _apply_ssl_view_augs(b2, patch=patch, mask_frac=mask_patch_frac)

        opt.zero_grad(set_to_none=True)

        with sdp_math_only():
            with torch.autocast(device_type=dev_type, enabled=bool(use_amp and dev_type == "cuda")):
                out1 = model.forward_ssl(b1)
                out2 = model.forward_ssl(b2)

            p1 = out1["proj"]
            p2 = out2["proj"]

            l_nce = _info_nce_symmetric(p1, p2, temp=temp)

            me12 = (out1["me"] * out2["me"]).detach()
            mm12 = (out1["mm"] * out2["mm"]).detach()
            mt12 = (out1["mt"] * out2["mt"]).detach()

            rec_e = 0.5 * (_masked_mse(out1["pred_e"], out2["He"].detach(), me12) + _masked_mse(out2["pred_e"], out1["He"].detach(), me12))
            rec_m = 0.5 * (_masked_mse(out1["pred_m"], out2["Hm"].detach(), mm12) + _masked_mse(out2["pred_m"], out1["Hm"].detach(), mm12))
            rec_t = 0.5 * (_masked_mse(out1["pred_t"], out2["Ht"].detach(), mt12) + _masked_mse(out2["pred_t"], out1["Ht"].detach(), mt12))

            l_rec = (rec_e + rec_m + rec_t) / 3.0
            loss = float(lambda_nce) * l_nce + float(lambda_rec) * l_rec

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
            opt.step()

        bs = int(batch["X_EEG"].shape[0])
        total_loss += float(loss.item()) * bs
        total_nce += float(l_nce.item()) * bs
        total_rec += float(l_rec.item()) * bs
        n_seen += bs

    return {
        "ssl_loss": total_loss / max(1, n_seen),
        "ssl_nce": total_nce / max(1, n_seen),
        "ssl_rec": total_rec / max(1, n_seen),
        "n": int(n_seen),
    }

# ============================================================
# Threshold tuning for ACTION head (supports per-scenario)
# ============================================================
def _binary_bal_acc(y_true01: np.ndarray, y_pred01: np.ndarray) -> float:
    y_true01 = np.asarray(y_true01).astype(np.int64)
    y_pred01 = np.asarray(y_pred01).astype(np.int64)
    pos = (y_true01 == 1)
    neg = ~pos
    if pos.sum() == 0 or neg.sum() == 0:
        return 0.0
    tpr = float((y_pred01[pos] == 1).mean())
    tnr = float((y_pred01[neg] == 0).mean())
    return 0.5 * (tpr + tnr)

def _binary_macro_f1(y_true01: np.ndarray, y_pred01: np.ndarray) -> float:
    rep = classification_report_np(y_true01, y_pred01, ["REST", "ACTION"])
    return float(rep["macro_f1"])


@torch.no_grad()
def tune_action_threshold_on_val_fast(
    model,
    va_loader,
    device: str,
    use_amp: bool,
    scenario: str = "S0",
    objective: str = "macroF1",
    n_thresholds: int = 31,
):
    model.eval()
    y_true, y_score = [], []

    skip = {
        "y_action","y_task_raw","task_valid","use_for_task",
        "subject_id","task_code","trial_id","win_type","qc_flag",
        "q_score","kept_modalities",
        "dist_to_onset_s","onset_idx_nearest","onset_time_nearest_s",
        "w_label_mean","active_prob_mean","is_transition"
    }
    dev_type = "cuda" if str(device).startswith("cuda") else "cpu"

    for batch in va_loader:
        # keep labels on CPU
        ya_cpu = batch["y_action"].numpy().astype(np.int64, copy=False)

        # move tensors to device (except skipped meta/labels)
        batch = to_device(batch, device, skip_keys=skip)
        apply_scenario_inplace(batch, scenario)

        with torch.autocast(device_type=dev_type, enabled=bool(use_amp and dev_type == "cuda")):
            logits_a, _, _, _ = model.forward_window(batch)

        # ✅ consistent with your eval style
        pa_t = torch.softmax(logits_a, dim=-1)[:, 1]  # (B,)
        pa   = pa_t.detach().cpu().numpy().astype(np.float32, copy=False)

        # ✅ ALWAYS append (not inside any condition)
        y_true.append(ya_cpu)
        y_score.append(pa)

    # ✅ guard against empty loader / append indentation bugs
    if len(y_true) == 0:
        raise RuntimeError("tune_action_threshold_on_val_fast: no batches appended (empty loader or indentation bug).")

    y_true  = np.concatenate(y_true, axis=0)
    y_score = np.concatenate(y_score, axis=0)

    # allow thresholds above 0.95 (needed for S2/S6 to remove REST FPs)
    thr_grid = np.linspace(0.05, 0.99, int(n_thresholds))
    thr_grid = np.unique(np.concatenate([thr_grid, np.array([0.97, 0.98, 0.99], dtype=np.float32)]))

    best_thr = 0.5
    best_obj = -1e9

    for thr in thr_grid:
        y_pred = (y_score >= thr).astype(np.int64)
        obj = _binary_bal_acc(y_true, y_pred) if objective == "bal_acc" else _binary_macro_f1(y_true, y_pred)
        if obj > best_obj:
            best_obj = float(obj)
            best_thr = float(thr)

    ap = binary_average_precision(y_true, y_score)
    return {
        "best_thr": float(best_thr),
        "objective": str(objective),
        "best_objective_value": float(best_obj),
        "val_auprc": float(ap),
        "n": int(y_true.size),
        "pos": int((y_true == 1).sum()),
        "neg": int((y_true == 0).sum()),
    }



def tune_action_threshold_commit_safe(
    model, va_loader, device: str, use_amp: bool, scenario: str,
    n_thresholds: int = 31, lambda_fpr: float = 3.0,
    thr_cap: Optional[float] = None,   # optional: cap thrCT (e.g., 0.97)
):
    model.eval()
    y_true, y_score = [], []
    dev_type = "cuda" if str(device).startswith("cuda") else "cpu"

    skip = {"y_action","y_task_raw","task_valid","use_for_task",
            "subject_id","task_code","trial_id","win_type","qc_flag",
            "q_score","kept_modalities","dist_to_onset_s","onset_idx_nearest",
            "onset_time_nearest_s","w_label_mean","active_prob_mean","is_transition"}

    for batch in va_loader:
        ya_cpu = batch["y_action"].numpy().astype(np.int64, copy=False)
        batch = to_device(batch, device, skip_keys=skip)
        apply_scenario_inplace(batch, scenario)

        with torch.autocast(device_type=dev_type, enabled=bool(use_amp and dev_type == "cuda")):
            logits_a, _, _, _ = model.forward_window(batch)

        pa = torch.softmax(logits_a, dim=-1)[:, 1].detach().cpu().numpy().astype(np.float32, copy=False)
        y_true.append(ya_cpu)
        y_score.append(pa)

    if len(y_true) == 0:
        raise RuntimeError("tune_action_threshold_commit_safe: empty va_loader (no batches).")

    y_true  = np.concatenate(y_true, axis=0)
    y_score = np.concatenate(y_score, axis=0)

    thr_grid = np.linspace(0.05, 0.99, int(n_thresholds), dtype=np.float64)
    thr_grid = np.unique(np.concatenate([thr_grid, np.array([0.97, 0.98, 0.99], dtype=np.float64)]))
    if thr_cap is not None:
        thr_grid = thr_grid[thr_grid <= float(thr_cap) + 1e-12]

    best_thr = 0.5
    best_obj = -1e18
    best_f1  = -1e9
    best_fpr = 1e9

    for thr in thr_grid:
        y_pred = (y_score >= thr).astype(np.int64)
        f1  = float(_binary_macro_f1(y_true, y_pred))
        fpr = float(rest_fp_rate_action(y_true, y_pred))

        obj = f1 - float(lambda_fpr) * fpr  # ✅ tradeoff

        # tie-breakers: higher F1, then lower FPR
        if (obj > best_obj + 1e-12) or (abs(obj - best_obj) <= 1e-12 and (f1 > best_f1 + 1e-12)) or \
           (abs(obj - best_obj) <= 1e-12 and abs(f1 - best_f1) <= 1e-12 and fpr < best_fpr):
            best_obj = obj
            best_thr = float(thr)
            best_f1  = f1
            best_fpr = fpr

    return float(best_thr)







# ============================================================
# Supervised loss (ACTION + TASK + COMMIT) + ✅ Dominance regularizer
# ============================================================
def _normalized_entropy_3(w: torch.Tensor, avail: torch.Tensor, eps: float = 1e-6) -> torch.Tensor:
    """
    w: (B,3) soft weights; avail: (B,3) binary availability
    returns normalized entropy in [0,1] (higher => less dominance)
    """
    w = w * avail
    w = w / w.sum(dim=1, keepdim=True).clamp_min(eps)
    H = -(w * (w + eps).log()).sum(dim=1)
    return H / math.log(3.0)

def supervised_loss_iros(
    logits_action: torch.Tensor,
    logits_task: torch.Tensor,
    ct: torch.Tensor,
    st: Dict[str, torch.Tensor],
    batch: Dict[str, torch.Tensor],
    w_task: Optional[torch.Tensor],
    alpha_task: float = 1.0,
    use_sample_weight: bool = False,
    alpha_commit: float = 1.0,
    alpha_false_commit_rest: float = 1.0,
    alpha_flap: float = 0.25,
):
    """
    Supervised loss = ACTION CE + alpha_task * TASK CE(masked) + COMMIT losses
    Updates:
      - uses ct_logit (BCEWithLogits) to avoid ct collapse
      - pos_weight computed from commit-supervised samples only
      - false-commit penalty applied ONLY when commit_mask==0 (no reliable supervision)
      - dominance regularizer uses RAW masks (M_EEG/M_EMG/M_ET) (or F_mask fallback)
    """
    ya = batch["y_action"].long()
    la = F.cross_entropy(logits_action, ya, reduction="none")  # (B,)

    # -------------------------
    # TASK loss (masked)
    # -------------------------
    task_mask = build_task_mask(batch)
    yt_raw = batch["y_task_raw"].long()
    ytK = (yt_raw - 1).clamp(min=0, max=logits_task.shape[-1] - 1)

    if w_task is not None:
        if not torch.is_tensor(w_task):
            w_task = torch.tensor(w_task, dtype=torch.float32)
        if w_task.device != logits_task.device:
            w_task = w_task.to(logits_task.device)
        if w_task.dtype != torch.float32:
            w_task = w_task.float()

    lt = torch.zeros_like(la)
    if task_mask.any():
        lt_raw = F.cross_entropy(
            logits_task[task_mask],
            ytK[task_mask],
            weight=w_task,
            reduction="none",
        )
        lt[task_mask] = lt_raw

    # -------------------------
    # COMMIT / readiness
    # -------------------------
    y_ready = batch.get("y_ready", batch["y_action"].float()).float().clamp(0, 1)          # (B,)
    commit_mask = batch.get("commit_mask", torch.ones_like(batch["y_action"]).float())     # (B,)
    commit_mask = commit_mask.float().clamp(0, 1)

    # We prefer logits for stability
    ct_logit = st.get("ct_logit", None)
    if ct_logit is None:
        # fallback (should not happen if you patched model to expose ct_logit)
        ct_logit = torch.logit(ct.float().clamp(1e-6, 1 - 1e-6))

    # pos_weight computed only on supervised samples (commit_mask>0)
    m = (commit_mask > 0).float()
    pos = ((y_ready > 0.5).float() * m).sum().clamp_min(1.0)
    neg = (((y_ready <= 0.5).float()) * m).sum().clamp_min(1.0)
    pos_w = (neg / pos).clamp(1.0, 20.0)

    # AMP-safe logits BCE
    ctx = (torch.autocast(device_type="cuda", enabled=False) if ct_logit.is_cuda else nullcontext())
    with ctx:
        l_ready = F.binary_cross_entropy_with_logits(
            ct_logit.float(),
            y_ready.float(),
            pos_weight=pos_w,
            reduction="none",
        )  # (B,)

    # supervise only where reliable
    l_ready = l_ready * commit_mask

    # Optional commit-state CE (masked) if logits exist
    if "commit_logits" in st:
        y_state = (y_ready >= 0.5).long()
        l_state = F.cross_entropy(st["commit_logits"], y_state, reduction="none") * commit_mask
    else:
        l_state = torch.zeros_like(l_ready)

    # False-commit penalty ONLY when no reliable commit supervision
    rest = (batch["y_action"].float() < 0.5).float()
    ct_prob = torch.sigmoid(ct_logit.float())

    # ✅ penalize ONLY when there is NO reliable supervision
    unsup = (commit_mask <= 0.0).float()
    l_false = rest * unsup * ct_prob


    # Flap regularizer (uses stability proxy)
    stab = st.get("delta_mean", None)
    l_flap = (ct_prob * stab.float()) if stab is not None else torch.zeros_like(l_ready)

    l_commit = (
        float(alpha_commit) * (l_ready + 0.25 * l_state)
        + float(alpha_false_commit_rest) * l_false
        + float(alpha_flap) * l_flap
    )

    per = la + float(alpha_task) * lt + l_commit

    # -------------------------
    # Modality dominance regularizer (entropy on fusion weights)
    # -------------------------
    dom_loss = torch.zeros_like(per)
    dom_H = None

    if DOM_REG_ENABLE and all(k in st for k in ["w_eeg", "w_emg", "w_et"]):
        w = torch.stack([st["w_eeg"], st["w_emg"], st["w_et"]], dim=1)  # (B,3)

        # availability from RAW masks (preferred)
        if ("M_EEG" in batch) and ("M_EMG" in batch) and ("M_ET" in batch):
            ae = (batch["M_EEG"].float().mean(dim=(1, 2)) > 0.05).float()
            am = (batch["M_EMG"].float().mean(dim=(1, 2)) > 0.05).float()
            at = (batch["M_ET"].float().mean(dim=(1, 2)) > 0.05).float()
            avail = torch.stack([ae, am, at], dim=1)  # (B,3)
        elif "F_mask" in batch:
            avail = (batch["F_mask"].float() > 0.5).float()  # (B,3)
        else:
            # fallback to st avail if present, else treat as available
            ae = st.get("avail_eeg", torch.ones_like(st["w_eeg"]))
            am = st.get("avail_emg", torch.ones_like(st["w_emg"]))
            at = st.get("avail_et",  torch.ones_like(st["w_et"]))
            avail = torch.stack([ae, am, at], dim=1)
            avail = (avail > 0.05).float()

        # apply reg only when >=2 modalities are available
        mask2 = (avail.sum(dim=1) >= 2).float()

        H = _normalized_entropy_3(w, avail)   # (B,)
        dom_H = H
        dom_pen = F.relu(float(DOM_MIN_ENTROPY) - H) * mask2
        dom_loss = float(DOM_REG_WEIGHT) * dom_pen
        per = per + dom_loss

    # -------------------------
    # Reduce (optionally sample-weighted)
    # -------------------------
    if use_sample_weight and "sample_weight" in batch:
        sw = batch["sample_weight"].float().clamp(min=0.0)
        loss = (sw * per).sum() / (sw.sum() + 1e-12)
    else:
        loss = per.mean()

    # diagnostics
    denom_cm = commit_mask.sum().clamp_min(1.0)
    diag = {
        "loss_action_mean": float(la.mean().item()),
        "loss_task_mean_masked": float(lt[task_mask].mean().item()) if task_mask.any() else 0.0,
        "task_mask_frac": float(task_mask.float().mean().item()),
        "loss_commit_mean": float(l_commit.mean().item()),
        "loss_ready_mean_masked": float((l_ready.sum() / denom_cm).item()),
        "commit_mask_mean": float(commit_mask.mean().item()),
        "false_commit_rest_mean": float(l_false.mean().item()),
        "ct_mean": float(ct_prob.mean().item()),
        "ct_rest_mean": float(ct_prob[batch["y_action"] == 0].mean().item()) if (batch["y_action"] == 0).any() else 0.0,
        "ct_action_mean": float(ct_prob[batch["y_action"] == 1].mean().item()) if (batch["y_action"] == 1).any() else 0.0,
        "pos_weight_commit": float(pos_w.detach().item()) if torch.is_tensor(pos_w) and pos_w.numel() == 1 else float("nan"),
        "dom_loss_mean": float(dom_loss.mean().item()) if torch.is_tensor(dom_loss) else 0.0,
        "dom_entropy_mean": float(dom_H.mean().item()) if (dom_H is not None) else float("nan"),
    }
    return loss, diag
def _train_one_epoch_impl(model, loader, opt, scaler, device, w_task, alpha_task,
                          use_sample_weight, grad_clip, use_amp, dev_type):
    total = 0.0
    n = 0
    agg = defaultdict(float)

    for batch in loader:
        batch = to_device(batch, device)
        batch = apply_train_augs(batch)

        # ✅ NEW: scenario mix (train like eval)
        batch = apply_train_scenario_mix_inplace(batch)


   
        # ✅ DEBUG ONCE (first batch of whole run)
        if (n == 0) and (not getattr(model, "_did_debug_batch", False)):
            debug_batch(batch)
            model._did_debug_batch = True



        opt.zero_grad(set_to_none=True)

        with sdp_math_only():
            with torch.autocast(device_type=dev_type, enabled=bool(use_amp and dev_type == "cuda")):
                logits_a, logits_t, ct, st = model.forward_window(batch)
                loss, diag = supervised_loss_iros(
                    logits_action=logits_a,
                    logits_task=logits_t,
                    ct=ct,
                    st=st,
                    batch=batch,
                    w_task=w_task,
                    alpha_task=float(alpha_task),
                    use_sample_weight=bool(use_sample_weight),
                    alpha_commit=2.00,
                    alpha_false_commit_rest=0.25,
                    alpha_flap=0.02,
                )

        if scaler is not None:
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
            scaler.step(opt)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), float(grad_clip))
            opt.step()

        bs = int(batch["y_action"].shape[0])
        total += float(loss.item()) * bs
        n += bs
        for k, v in diag.items():
            agg[k] += float(v) * bs

    out = {"train_loss": total / max(1, n)}
    for k in agg:
        out[k] = agg[k] / max(1, n)
    return out


def train_one_epoch(
    model,
    loader,
    opt,
    scaler,
    device: str,
    w_task: Optional[torch.Tensor],
    alpha_task: float = 1.0,
    use_sample_weight: bool = False,
    grad_clip: float = 1.0,
    use_amp: bool = True,
):
    model.train()
    dev_type = "cuda" if str(device).startswith("cuda") else "cpu"
    return _train_one_epoch_impl(
        model, loader, opt, scaler, device, w_task, alpha_task,
        use_sample_weight, grad_clip, use_amp, dev_type
    )

# ============================================================
# Task class weights (same as before)
# ============================================================
def compute_task_class_weights_from_shards(ds_tr: ShardNPZDataset, num_task: int = 5):
    task_counts = np.zeros((num_task,), dtype=np.int64)

    for sp in ds_tr.shard_paths:
        with np.load(sp, allow_pickle=False) as z:
            ya = np.asarray(z["y_action"]).astype(np.int64, copy=False)

            if "y_task" in z.files:
                yt = np.asarray(z["y_task"]).astype(np.int64, copy=False)
            elif "y_task_raw" in z.files:
                yt = np.asarray(z["y_task_raw"]).astype(np.int64, copy=False)
            else:
                yt = np.zeros_like(ya)

            tv = np.asarray(z["task_valid"]).astype(np.int64, copy=False) if "task_valid" in z.files else np.ones_like(ya)
            uft = np.asarray(z["use_for_task"]).astype(np.int64, copy=False) if "use_for_task" in z.files else np.ones_like(ya)

        m = (ya == 1) & (tv == 1) & (uft == 1) & (yt > 0)
        if m.any():
            yk = yt[m] - 1
            for k in range(num_task):
                task_counts[k] += int((yk == k).sum())

    task_counts = np.maximum(task_counts, 1)
    w_task = (task_counts.sum() / task_counts).astype(np.float32)
    w_task = w_task / w_task.mean()
    return w_task, task_counts


# ============================================================
# Commit metrics (IROS decision quality)
# ============================================================
def compute_commit_event_metrics(
    y_action_true: np.ndarray,
    ct: np.ndarray,
    cfg: NeuroCommitCfg,
    stride_s: float = 0.25,
) -> Dict[str, float]:
    """
    Computes event-rate style commit metrics over a stream.
    NOTE: If your loader mixes trials, this is an approximation. For best results,
    compute per-trial streams using (subject_id, task_code, trial_id) grouping when available.
    """
    filt = CommitDecisionFilter(
        thr_on=cfg.commit_thr_on,
        thr_off=cfg.commit_thr_off,
        dwell_windows=cfg.commit_dwell_windows,
        cooldown_windows=cfg.commit_cooldown_windows,
    )
    state_seq = []
    commit_events = []
    for t in range(len(ct)):
        st, ev = filt.step(float(ct[t]))
        state_seq.append(st)
        commit_events.append(1 if ev else 0)

    state_seq = np.asarray(state_seq)
    commit_events = np.asarray(commit_events).astype(np.int64)
    y_action_true = np.asarray(y_action_true).astype(np.int64)

    rest = (y_action_true == 0)
    action = (y_action_true == 1)

    false_commit_events = int(((commit_events == 1) & rest).sum())
    rest_dur_s = float(rest.sum()) * float(stride_s)
    false_commits_per_hour = (false_commit_events / max(1e-9, rest_dur_s)) * 3600.0

    changes = int((state_seq[1:] != state_seq[:-1]).sum()) if len(state_seq) > 1 else 0
    total_dur_s = float(len(state_seq)) * float(stride_s)
    flaps_per_min = (changes / max(1e-9, total_dur_s)) * 60.0

    ct_rest = float(ct[rest].mean()) if rest.any() else float("nan")
    ct_action = float(ct[action].mean()) if action.any() else float("nan")

    return {
        "false_commits_per_hour_rest": float(false_commits_per_hour),
        "flaps_per_min": float(flaps_per_min),
        "ct_mean": float(np.mean(ct)) if len(ct) else 0.0,
        "ct_rest_mean": ct_rest,
        "ct_action_mean": ct_action,
        "n": int(len(ct)),
        "false_commit_events": int(false_commit_events),
        "total_state_changes": int(changes),
    }

def time_to_commit_after_onset_if_available(
    df_rows: List[Dict[str, Any]],
    cfg: NeuroCommitCfg,
) -> float:
    """
    If metadata has (subject_id, task_code, trial_id, dist_to_onset_s),
    compute mean time-to-first-commit for windows with dist_to_onset_s>=0.
    If not available, returns NaN.
    """
    needed = {"subject_id", "task_code", "trial_id", "dist_to_onset_s", "ct"}
    if not all(all(k in r for k in needed) for r in df_rows):
        return float("nan")

    df = pd.DataFrame(df_rows)
    df = df[np.isfinite(df["dist_to_onset_s"].astype(float))].copy()
    df["dist_to_onset_s"] = df["dist_to_onset_s"].astype(float)
    df = df[df["dist_to_onset_s"] >= 0.0]

    if df.empty:
        return float("nan")

    out_times = []
    for (_, _, _), g in df.groupby(["subject_id", "task_code", "trial_id"]):
        g = g.sort_values("dist_to_onset_s")
        filt = CommitDecisionFilter(
            thr_on=cfg.commit_thr_on,
            thr_off=cfg.commit_thr_off,
            dwell_windows=cfg.commit_dwell_windows,
            cooldown_windows=cfg.commit_cooldown_windows,
        )
        first = None
        for _, row in g.iterrows():
            _, ev = filt.step(float(row["ct"]))
            if ev:
                first = float(row["dist_to_onset_s"])
                break
        if first is not None:
            out_times.append(first)
    return float(np.mean(out_times)) if len(out_times) else float("nan")
def compute_commit_metrics_per_trial(
    stream_rows: List[Dict[str, Any]],
    cfg: NeuroCommitCfg,
    stride_s: float = 0.25,
    gap_reset_s: Optional[float] = None,
) -> Dict[str, float]:
    """
    Per-trial commit metrics:
      - resets CommitDecisionFilter per (subject_id, task_code, trial_id)
      - sorts within trial by best available key
      - robust denominators for balanced exports:
          * false/hr at REST (observed)
          * false/1000 REST windows (stable, recommended for balanced exports)
          * false/hr total time (stable)
      - optional gap_reset_s: if timestamps jump too much, reset filter (avoid fake flaps across gaps)
    """
    if not stream_rows:
        return {
            "false_commits_per_hour_rest_observed": float("nan"),
            "false_commits_per_1000_rest_windows": float("nan"),
            "false_commits_per_hour_total": float("nan"),
            "flaps_per_min_observed": float("nan"),
            "ct_mean": float("nan"),
            "ct_rest_mean": float("nan"),
            "ct_action_mean": float("nan"),
            "n": 0,
            "n_rest_windows": 0,
            "n_action_windows": 0,
            "rest_dur_s_obs": 0.0,
            "total_dur_s_obs": 0.0,
            "false_commit_events": 0,
            "total_state_changes": 0,
        }

    gap_reset_s = float(gap_reset_s) if gap_reset_s is not None else float(2.5 * stride_s)

    df = pd.DataFrame(stream_rows).copy()

    # Prefer: center_time_s; else derive from onset_time+dist; else from win_idx*stride; else _ord
    if ("center_time_s" not in df.columns) and ("onset_time_nearest_s" in df.columns) and ("dist_to_onset_s" in df.columns):
        ot = pd.to_numeric(df["onset_time_nearest_s"], errors="coerce")
        dd = pd.to_numeric(df["dist_to_onset_s"], errors="coerce")
        df["center_time_s"] = ot + dd

    if "center_time_s" not in df.columns and "win_idx" in df.columns:
        wi = pd.to_numeric(df["win_idx"], errors="coerce")
        df["center_time_s"] = wi * float(stride_s)

    if "_ord" not in df.columns:
        df["_ord"] = np.arange(len(df), dtype=np.int64)

    key_cols = [c for c in ["subject_id", "task_code", "trial_id"] if c in df.columns]
    if len(key_cols) < 3:
        # fallback: treat as one stream
        key_cols = ["__all__"]
        df["__all__"] = 0

    sort_candidates = ["center_time_s", "win_idx", "onset_time_nearest_s", "dist_to_onset_s", "_ord"]
    sort_key = next((c for c in sort_candidates if c in df.columns), "_ord")
        
    # Prefer df["ct"] ALWAYS (already hard-gated in eval)
    if "ct" in df.columns:
        ct_use_all = pd.to_numeric(df["ct"], errors="coerce").to_numpy(np.float32, copy=False)
    elif ("ct_raw" in df.columns) and ("pa" in df.columns):
        ct_use_all = pd.to_numeric(df["ct_raw"], errors="coerce").to_numpy(np.float32, copy=False) * \
                     pd.to_numeric(df["pa"], errors="coerce").to_numpy(np.float32, copy=False)
    else:
        raise RuntimeError("compute_commit_metrics_per_trial: missing ct (and no ct_raw/pa fallback).")



    # y_action
    y_all = pd.to_numeric(df["y_action"], errors="coerce").fillna(0).astype(np.int64).to_numpy(copy=False)

    # stats accumulators
    total_false_events = 0
    total_state_changes = 0

    n_rest = 0
    n_action = 0
    n_total = 0

    # "observed" durations (window-count based; valid for same export type)
    total_rest_dur_s_obs = 0.0
    total_dur_s_obs = 0.0

    # optional timeline-based duration (only used for extra diagnostics; not relied on for paper)
    total_dur_s_timeline = 0.0
    total_rest_dur_s_timeline = 0.0

    # group per trial
    for _, g in df.groupby(key_cols):
        g = g.sort_values(sort_key)

        idx = g.index.to_numpy()
        y = y_all[df.index.get_indexer(idx)]
        ct_use = ct_use_all[df.index.get_indexer(idx)].astype(np.float32, copy=False)

        # duration estimates
        t = pd.to_numeric(g[sort_key], errors="coerce").to_numpy(np.float64, copy=False)
        # dt between successive samples (fallback to stride_s if invalid)
        dt = np.diff(t, append=(t[-1] + float(stride_s) if len(t) else float(stride_s)))
        dt = np.where(np.isfinite(dt) & (dt > 0), dt, float(stride_s))

        # observed durations
        n_total += int(len(y))
        n_rest += int((y == 0).sum())
        n_action += int((y == 1).sum())

        total_dur_s_obs += float(len(y)) * float(stride_s)
        total_rest_dur_s_obs += float((y == 0).sum()) * float(stride_s)

        # timeline durations (diagnostic only)
        total_dur_s_timeline += float(dt.sum())
        total_rest_dur_s_timeline += float(dt[(y == 0)].sum()) if len(dt) else 0.0

        # run filter with optional gap resets
        filt = CommitDecisionFilter(
            thr_on=cfg.commit_thr_on,
            thr_off=cfg.commit_thr_off,
            dwell_windows=cfg.commit_dwell_windows,
            cooldown_windows=cfg.commit_cooldown_windows,
        )

        prev_state = filt.state
        for i in range(len(ct_use)):
            # if there's a big gap before this step, reset (avoid fake flaps across discontinuity)
            if i > 0 and float(dt[i - 1]) > gap_reset_s:
                filt.reset()
                prev_state = filt.state

            st, ev = filt.step(float(np.clip(ct_use[i], 0.0, 1.0)))

            if ev and int(y[i]) == 0:
                total_false_events += 1

            if st != prev_state:
                total_state_changes += 1
            prev_state = st

    # rates
    false_per_hr_rest_obs = (total_false_events / max(1e-9, total_rest_dur_s_obs)) * 3600.0
    false_per_1k_rest = (total_false_events / max(1, n_rest)) * 1000.0
    false_per_hr_total = (total_false_events / max(1e-9, total_dur_s_obs)) * 3600.0

    flaps_per_min_obs = (total_state_changes / max(1e-9, total_dur_s_obs)) * 60.0

    # ct means (use ct_use_all)
    ct_mean = float(np.nanmean(ct_use_all)) if ct_use_all.size else float("nan")
    ct_rest_mean = float(np.nanmean(ct_use_all[y_all == 0])) if np.any(y_all == 0) else float("nan")
    ct_action_mean = float(np.nanmean(ct_use_all[y_all == 1])) if np.any(y_all == 1) else float("nan")

    return {
        # ✅ paper-safe + stable
        "false_commits_per_hour_rest_observed": float(false_per_hr_rest_obs),
        "false_commits_per_1000_rest_windows": float(false_per_1k_rest),
        "false_commits_per_hour_total": float(false_per_hr_total),
        "flaps_per_min_observed": float(flaps_per_min_obs),

        # ct summaries
        "ct_mean": ct_mean,
        "ct_rest_mean": ct_rest_mean,
        "ct_action_mean": ct_action_mean,

        # denominators + counts (for reporting/sanity checks)
        "n": int(n_total),
        "n_rest_windows": int(n_rest),
        "n_action_windows": int(n_action),
        "rest_dur_s_obs": float(total_rest_dur_s_obs),
        "total_dur_s_obs": float(total_dur_s_obs),

        # diagnostics (optional)
        "rest_dur_s_timeline": float(total_rest_dur_s_timeline),
        "total_dur_s_timeline": float(total_dur_s_timeline),

        # events / changes
        "false_commit_events": int(total_false_events),
        "total_state_changes": int(total_state_changes),
    }


# ============================================================
# Eval (ACTION/TASK + COMMIT metrics + optional preds/stream)
# ============================================================
def _np_percentiles(x: np.ndarray, ps=(5, 50, 95)):
    x = np.asarray(x, dtype=np.float64).reshape(-1)
    if x.size == 0:
        return {f"p{p:02d}": float("nan") for p in ps}
    return {f"p{p:02d}": float(np.percentile(x, p)) for p in ps}

@torch.no_grad()
def eval_one_split_fast(
    model,
    loader,
    device,
    use_amp: bool,
    scenario: str,
    action_threshold: float,
    cfgM: NeuroCommitCfg,
    robustness: Optional[str] = None,
    save_preds: bool = False,
    save_stream: bool = False,
    commit_gate_threshold: Optional[float] = None,   # ✅ NEW
):

    model.eval()
    save_stream = bool(save_stream)

    yA_true, yA_pred, yA_score = [], [], []
    yT_true, yT_pred, yT_prob = [], [], []

    ct_all = []
    w_e_all, w_m_all, w_t_all = [], [], []
    a_e_all, a_m_all, a_t_all = [], [], []

    stream_rows = []

    preds_pack = None
    if save_preds:
        preds_pack = {
            "y_action_true": [], "y_action_pred": [], "y_action_score": [],
            "y_task_trueK": [], "y_task_predK": [],
            "ct": [],
        }

    dev_type = "cuda" if str(device).startswith("cuda") else "cpu"

    for batch in loader:
        # keep labels/meta on CPU
        ya_cpu = batch["y_action"].numpy().astype(np.int64, copy=False)
        yt_cpu = batch["y_task_raw"].numpy().astype(np.int64, copy=False)
        tv_cpu = batch.get("task_valid", torch.ones_like(batch["y_action"])).numpy().astype(np.int64, copy=False)
        uft_cpu = batch.get("use_for_task", torch.ones_like(batch["y_action"])).numpy().astype(np.int64, copy=False)

        meta_cpu = {}
        for mk in [
            "subject_id", "task_code", "trial_id",
            "dist_to_onset_s", "onset_time_nearest_s",
            "win_idx", "win_type", "q_score", "qc_flag"
        ]:
            if mk in batch:
                meta_cpu[mk] = batch[mk]

        skip = {
            "y_action", "y_task_raw", "task_valid", "use_for_task",
            "subject_id", "task_code", "trial_id", "win_type",
            "qc_flag", "q_score", "kept_modalities",
            "dist_to_onset_s", "onset_idx_nearest", "onset_time_nearest_s",
            "w_label_mean", "active_prob_mean", "is_transition"
        }
        batch = to_device(batch, device, skip_keys=skip)

        apply_scenario_inplace(batch, scenario)
        if robustness == "R3":
            apply_R3_partial_time_dropout_inplace(batch, chunk_frac=0.20, p=1.0)
        elif robustness == "R4":
            apply_R4_noise_artifact_inplace(batch, sigma=0.05, p=1.0)

        with torch.autocast(device_type=dev_type, enabled=bool(use_amp and dev_type == "cuda")):
            logits_a, logits_t, ct, st = model.forward_window(batch)

        # -----------------------------
        # ✅ ACTION prob + prediction
        # -----------------------------
        pa_t = torch.softmax(logits_a, dim=-1)[:, 1]  # (B,)
        pa = pa_t.detach().cpu().numpy().astype(np.float32, copy=False)
        predA = (pa >= float(action_threshold)).astype(np.int64)

        ct_raw = ct.detach().cpu().numpy().astype(np.float32, copy=False)

        # ACTION threshold (for action metrics)
        predA = (pa >= float(action_threshold)).astype(np.int64)

        # ✅ Separate threshold used ONLY to gate CT for commit decisions
        thr_gate = float(action_threshold if commit_gate_threshold is None else commit_gate_threshold)
        predA_gate_t = (pa_t >= thr_gate).float()

        ct_raw = ct.detach().cpu().numpy().astype(np.float32, copy=False)
        ct_eff_t = (ct * predA_gate_t).clamp(0.0, 1.0)
        ct_eff = ct_eff_t.detach().cpu().numpy().astype(np.float32, copy=False)




        # ✅ THESE APPENDS MUST ALWAYS RUN (fixes concatenate error)
        yA_true.append(ya_cpu)
        yA_pred.append(predA)
        yA_score.append(pa)
        ct_all.append(ct_eff)

        # gate/avail stats
        if isinstance(st, dict) and all(k in st for k in ["w_eeg", "w_emg", "w_et", "avail_eeg", "avail_emg", "avail_et"]):
            w_e_all.append(st["w_eeg"].detach().cpu().numpy())
            w_m_all.append(st["w_emg"].detach().cpu().numpy())
            w_t_all.append(st["w_et"].detach().cpu().numpy())
            a_e_all.append(st["avail_eeg"].detach().cpu().numpy())
            a_m_all.append(st["avail_emg"].detach().cpu().numpy())
            a_t_all.append(st["avail_et"].detach().cpu().numpy())

        # stream rows (for per-trial commit metrics + onset metrics)
        if save_stream and meta_cpu:
            B = int(len(ya_cpu))
            w_e_b = st["w_eeg"].detach().cpu().numpy() if ("w_eeg" in st) else None
            w_m_b = st["w_emg"].detach().cpu().numpy() if ("w_emg" in st) else None
            w_t_b = st["w_et"].detach().cpu().numpy() if ("w_et" in st) else None

            for i in range(B):
                row = {
                    "ct": float(ct_eff[i]),      # ✅ gated ct for event metrics
                    "ct_raw": float(ct_raw[i]),  # optional debug
                    "pa": float(pa[i]),          # optional debug
                    "y_action": int(ya_cpu[i]),
                    "_ord": int(len(stream_rows)),
                }
                


                if w_e_b is not None: row["w_eeg"] = float(w_e_b[i])
                if w_m_b is not None: row["w_emg"] = float(w_m_b[i])
                if w_t_b is not None: row["w_et"]  = float(w_t_b[i])

                for mk, mv in meta_cpu.items():
                    try:
                        v = mv[i]
                    except Exception:
                        v = mv
                    if torch.is_tensor(v):
                        v = v.item() if v.numel() == 1 else v.detach().cpu().numpy()
                    elif isinstance(v, np.generic):
                        v = v.item()
                    elif isinstance(v, (bytes, bytearray)):
                        v = v.decode("utf-8", errors="ignore")
                    row[mk] = v

                # reconstruct center_time_s for ordering
                if ("onset_time_nearest_s" in row) and ("dist_to_onset_s" in row):
                    try:
                        ot = float(row["onset_time_nearest_s"])
                        dd = float(row["dist_to_onset_s"])
                        if np.isfinite(ot) and np.isfinite(dd):
                            row["center_time_s"] = ot + dd
                    except Exception:
                        pass

                stream_rows.append(row)

        # TASK head only on valid action windows
        m_task = (ya_cpu == 1) & (tv_cpu == 1) & (uft_cpu == 1) & (yt_cpu > 0)
        if m_task.any():
            idx_np = np.where(m_task)[0].astype(np.int64)
            idx_t = torch.from_numpy(idx_np).to(logits_t.device)

            probT = torch.softmax(logits_t[idx_t], dim=-1).detach().cpu().numpy()
            predT = np.argmax(probT, axis=-1).astype(np.int64)
            ytK = (yt_cpu[m_task] - 1).astype(np.int64)

            yT_true.append(ytK)
            yT_pred.append(predT)
            yT_prob.append(probT)

            if save_preds:
                preds_pack["y_task_trueK"].append(ytK)
                preds_pack["y_task_predK"].append(predT)


        if save_preds:
            preds_pack["y_action_true"].append(ya_cpu)
            preds_pack["y_action_pred"].append(predA)
            preds_pack["y_action_score"].append(pa)
            preds_pack["ct"].append(ct_eff)


    # ✅ guard against empty loader / indentation bugs
    if len(yA_true) == 0:
        raise RuntimeError(
            "eval_one_split_fast: no batches appended (empty loader or indentation bug around yA_true.append/yA_pred.append/yA_score.append)."
        )

    # concat
    yA_true  = np.concatenate(yA_true, axis=0)
    yA_pred  = np.concatenate(yA_pred, axis=0)
    yA_score = np.concatenate(yA_score, axis=0)
    ct_all   = np.concatenate(ct_all, axis=0) if len(ct_all) else np.asarray([], dtype=np.float32)

    if len(yT_true) > 0:
        yT_true = np.concatenate(yT_true, axis=0)
        yT_pred = np.concatenate(yT_pred, axis=0)
        yT_prob = np.concatenate(yT_prob, axis=0)
    else:
        yT_true = np.asarray([], dtype=np.int64)
        yT_pred = np.asarray([], dtype=np.int64)
        yT_prob = np.zeros((0, 5), dtype=np.float32)

    ACTION_NAMES = ["REST", "ACTION"]
    TASK_NAMES   = ["T1", "T2", "T3", "T4", "T5"]

    repA = classification_report_np(yA_true, yA_pred, ACTION_NAMES)
    repA["auprc"] = float(binary_average_precision(yA_true, yA_score))
    repA["rest_fp_rate"] = float(rest_fp_rate_action(yA_true, yA_pred))

    if yT_true.size > 0:
        repT = classification_report_np(yT_true, yT_pred, TASK_NAMES)
        repT["auprc"] = float(multiclass_macro_auprc_ovr(yT_true, yT_prob, num_class=5))
    else:
        repT = {
            "acc": 0.0, "bal_acc": 0.0, "macro_precision": 0.0, "macro_recall": 0.0, "macro_f1": 0.0,
            "cm": np.zeros((5, 5), dtype=np.int64),
            "rows": [{"class_id": i, "class_name": TASK_NAMES[i], "precision": 0.0, "recall": 0.0, "f1": 0.0,
                      "acc_class": 0.0, "specificity": 0.0, "support": 0} for i in range(5)],
            "auprc": 0.0,
        }

    # commit metrics
    if save_stream and len(stream_rows) > 0:
        commit_metrics = compute_commit_metrics_per_trial(stream_rows=stream_rows, cfg=cfgM, stride_s=float(WIN_STRIDE_S))
        ttc_mean = time_to_commit_after_onset_if_available(stream_rows, cfgM)
        commit_metrics["time_to_commit_mean_s"] = float(ttc_mean) if np.isfinite(ttc_mean) else float("nan")
    else:
        commit_metrics = compute_commit_event_metrics(y_action_true=yA_true, ct=ct_all, cfg=cfgM, stride_s=float(WIN_STRIDE_S))
        commit_metrics["time_to_commit_mean_s"] = float("nan")

    # gate/avail summary
    def _safe_mean(xlist):
        if not xlist:
            return float("nan")
        x = np.concatenate([np.asarray(a).reshape(-1) for a in xlist], axis=0)
        return float(np.mean(x)) if x.size else float("nan")

    # -------------------------
    # extras (log everything you need for paper + stats)
    # -------------------------
    extras = {
        # -------------------------
        # ACTION
        # -------------------------
        "action_rest_fpr": float(repA.get("rest_fp_rate", 0.0)),
        "action_threshold": float(action_threshold),

        # ✅ add this if you introduced separate commit gating
        # (pass commit_gate_threshold into eval_one_split_fast and store it here)
        "commit_gate_threshold": float(commit_gate_threshold) if (commit_gate_threshold is not None) else float("nan"),

        "robustness": robustness or "",
        "n_action": int(yA_true.size),
        "n_task": int(yT_true.size),

        # -------------------------
        # ✅ COMMIT metrics (robust + paper-safe)
        # -------------------------
        # stable + recommended for balanced exports
        "commit_false_per_1000_rest": float(commit_metrics.get("false_commits_per_1000_rest_windows", float("nan"))),

        # still report (can inflate if REST is downsampled)
        "commit_false_per_hour_rest": float(
            commit_metrics.get(
                "false_commits_per_hour_rest_observed",
                commit_metrics.get("false_commits_per_hour_rest", float("nan"))
            )
        ),

        # stable overall rate
        "commit_false_per_hour_total": float(commit_metrics.get("false_commits_per_hour_total", float("nan"))),

        # flaps
        "commit_flaps_per_min": float(
            commit_metrics.get(
                "flaps_per_min_observed",
                commit_metrics.get("flaps_per_min", float("nan"))
            )
        ),

        # denominators / counts (debugging + paper)
        "commit_false_events": int(commit_metrics.get("false_commit_events", 0)),
        "commit_state_changes": int(commit_metrics.get("total_state_changes", 0)),
        "commit_n_rest": int(commit_metrics.get("n_rest_windows", 0)),
        "commit_n_action": int(commit_metrics.get("n_action_windows", 0)),
        "commit_rest_dur_s_obs": float(commit_metrics.get("rest_dur_s_obs", float("nan"))),
        "commit_total_dur_s_obs": float(commit_metrics.get("total_dur_s_obs", float("nan"))),

        # ct means
        "ct_mean": float(commit_metrics.get("ct_mean", float("nan"))),
        "ct_rest_mean": float(commit_metrics.get("ct_rest_mean", float("nan")))
            if np.isfinite(float(commit_metrics.get("ct_rest_mean", float("nan")))) else float("nan"),
        "ct_action_mean": float(commit_metrics.get("ct_action_mean", float("nan")))
            if np.isfinite(float(commit_metrics.get("ct_action_mean", float("nan")))) else float("nan"),

        # onset/ttc (if available)
        "time_to_commit_mean_s": float(commit_metrics.get("time_to_commit_mean_s", float("nan")))
            if np.isfinite(float(commit_metrics.get("time_to_commit_mean_s", float("nan")))) else float("nan"),

        # -------------------------
        # ct percentiles (use ct_all stream you already built)
        # -------------------------
        **{f"ct_{k}": float(v) for k, v in _np_percentiles(ct_all).items()},
        **{f"ct_rest_{k}": float(v) for k, v in _np_percentiles(ct_all[yA_true == 0]).items()},
        **{f"ct_action_{k}": float(v) for k, v in _np_percentiles(ct_all[yA_true == 1]).items()},

        # -------------------------
        # gate means (overall)
        # -------------------------
        "w_eeg_mean": float(_safe_mean(w_e_all)),
        "w_emg_mean": float(_safe_mean(w_m_all)),
        "w_et_mean":  float(_safe_mean(w_t_all)),
        "avail_eeg_mean": float(_safe_mean(a_e_all)),
        "avail_emg_mean": float(_safe_mean(a_m_all)),
        "avail_et_mean":  float(_safe_mean(a_t_all)),
    }

    # -------------------------
    # preds pack concat (unchanged)
    # -------------------------
    if save_preds and preds_pack is not None:
        for k in ["y_action_true", "y_action_pred", "y_action_score", "ct", "y_task_trueK", "y_task_predK"]:
            preds_pack[k] = np.concatenate(preds_pack[k], axis=0) if len(preds_pack[k]) else np.asarray([])

    if save_stream:
        return repA, repT, extras, preds_pack, stream_rows
    return repA, repT, extras, preds_pack


# ============================================================
# Writers
# ============================================================
def _save_cm_csv(cm: np.ndarray, out_csv: Path, row_names: List[str], col_names: List[str]):
    df = pd.DataFrame(cm.astype(int), index=row_names, columns=col_names)
    df.to_csv(out_csv, index=True)
def append_tidy_row(
    tidy_rows: list,
    tag: str, fold: int, split: str, scenario: str,
    label_frac: float,
    repA: dict, repT: dict, extras: dict,
):
    tidy_rows.append({
        "tag": str(tag),
        "fold": int(fold),
        "split": str(split),
        "scenario": str(scenario),
        "robustness": str(extras.get("robustness", "") or ""),
        "label_frac": float(label_frac),

        # -------------------------
        # ACTION metrics
        # -------------------------
        "action_acc": float(repA.get("acc", float("nan"))),
        "action_bal_acc": float(repA.get("bal_acc", float("nan"))),
        "action_macro_precision": float(repA.get("macro_precision", float("nan"))),
        "action_macro_recall": float(repA.get("macro_recall", float("nan"))),
        "action_macroF1": float(repA.get("macro_f1", float("nan"))),
        "action_AUPRC": float(repA.get("auprc", 0.0)),
        "action_rest_fpr": float(extras.get("action_rest_fpr", 0.0)),

        # -------------------------
        # TASK metrics
        # -------------------------
        "task_acc": float(repT.get("acc", float("nan"))),
        "task_bal_acc": float(repT.get("bal_acc", float("nan"))),
        "task_macro_precision": float(repT.get("macro_precision", float("nan"))),
        "task_macro_recall": float(repT.get("macro_recall", float("nan"))),
        "task_macroF1": float(repT.get("macro_f1", float("nan"))),
        "task_AUPRC": float(repT.get("auprc", 0.0)),

        # -------------------------
        # Thresholds + counts
        # -------------------------
        "action_threshold": float(extras.get("action_threshold", 0.5)),
        # ✅ if you added separate commit-gate threshold to extras, keep it; else NaN
        "commit_gate_threshold": float(extras.get("commit_gate_threshold", float("nan"))),

        "n_action": int(extras.get("n_action", 0)),
        "n_task": int(extras.get("n_task", 0)),

        # -------------------------
        # COMMIT metrics (paper-safe)
        # -------------------------
        # ✅ stable primary metric for balanced exports
        "commit_false_per_1000_rest": float(extras.get("commit_false_per_1000_rest", float("nan"))),

        # still useful to report (but can be misleading if REST is downsampled)
        "commit_false_per_hour_rest": float(extras.get("commit_false_per_hour_rest", float("nan"))),

        # ✅ stable overall rate
        "commit_false_per_hour_total": float(extras.get("commit_false_per_hour_total", float("nan"))),

        "commit_flaps_per_min": float(extras.get("commit_flaps_per_min", float("nan"))),

        # ✅ denominators + event counts (needed for sanity + paper)
        "commit_false_events": int(extras.get("commit_false_events", 0)),
        "commit_state_changes": int(extras.get("commit_state_changes", 0)),
        "commit_n_rest": int(extras.get("commit_n_rest", 0)),
        "commit_n_action": int(extras.get("commit_n_action", 0)),

        # ct means + TTC
        "ct_mean": float(extras.get("ct_mean", float("nan"))),
        "ct_rest_mean": float(extras.get("ct_rest_mean", float("nan"))),
        "ct_action_mean": float(extras.get("ct_action_mean", float("nan"))),
        "time_to_commit_mean_s": float(extras.get("time_to_commit_mean_s", float("nan"))),

        # ct percentiles
        "ct_p05": float(extras.get("ct_p05", float("nan"))),
        "ct_p50": float(extras.get("ct_p50", float("nan"))),
        "ct_p95": float(extras.get("ct_p95", float("nan"))),

        # gate/avail means
        "w_eeg_mean": float(extras.get("w_eeg_mean", float("nan"))),
        "w_emg_mean": float(extras.get("w_emg_mean", float("nan"))),
        "w_et_mean":  float(extras.get("w_et_mean",  float("nan"))),
        "avail_eeg_mean": float(extras.get("avail_eeg_mean", float("nan"))),
        "avail_emg_mean": float(extras.get("avail_emg_mean", float("nan"))),
        "avail_et_mean":  float(extras.get("avail_et_mean",  float("nan"))),
    })


def append_perclass_rows(
    perclass_rows: list,
    tag: str, fold: int, split: str, scenario: str,
    label_frac: float,
    head: str, rep: dict,
    robustness: str = "",
):
    for r in rep["rows"]:
        perclass_rows.append({
            "tag": tag,
            "fold": int(fold),
            "split": split,
            "scenario": scenario,
            "robustness": robustness,
            "label_frac": float(label_frac),
            "head": head,
            **r
        })


# ============================================================
# Eval + log + save helper
# ============================================================
def eval_log_save_split(
    *,
    model,
    loader,
    device: str,
    use_amp: bool,
    fold: int,
    split: str,
    scenario: str,
    action_threshold: float,
    commit_gate_threshold: Optional[float] = None,   # ✅ NEW
    cfgM: NeuroCommitCfg,
    out_dir: Path,
    tidy_rows: list,
    perclass_rows: list,
    tag: str,
    label_frac: float,
    robustness: Optional[str] = None,
    save_preds: bool = False,
    save_stream_csv: bool = True,
    save_cm_csv: bool = True,
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"_{robustness}" if robustness else ""

    out = eval_one_split_fast(
        model=model,
        loader=loader,
        device=device,
        use_amp=use_amp,
        scenario=scenario,
        action_threshold=float(action_threshold),
        commit_gate_threshold=(float(commit_gate_threshold) if commit_gate_threshold is not None else None),  # ✅ NEW
        cfgM=cfgM,
        robustness=robustness,
        save_preds=bool(save_preds),
        save_stream=True,
    )

    repA, repT, extras, preds_pack, stream_rows = out

    append_tidy_row(
        tidy_rows=tidy_rows,
        tag=str(tag),
        fold=int(fold),
        split=str(split),
        scenario=str(scenario),
        label_frac=float(label_frac),
        repA=repA,
        repT=repT,
        extras=extras,
    )
    append_perclass_rows(
        perclass_rows=perclass_rows,
        tag=str(tag),
        fold=int(fold),
        split=str(split),
        scenario=str(scenario),
        label_frac=float(label_frac),
        head="action",
        rep=repA,
        robustness=extras.get("robustness", "") or (robustness or ""),
    )
    append_perclass_rows(
        perclass_rows=perclass_rows,
        tag=str(tag),
        fold=int(fold),
        split=str(split),
        scenario=str(scenario),
        label_frac=float(label_frac),
        head="task",
        rep=repT,
        robustness=extras.get("robustness", "") or (robustness or ""),
    )

    if save_cm_csv:
        ACTION_NAMES = ["REST", "ACTION"]
        TASK_NAMES   = ["T1", "T2", "T3", "T4", "T5"]
        cmA_path = out_dir / f"cm_{split}_{scenario}{suffix}_action.csv"
        cmT_path = out_dir / f"cm_{split}_{scenario}{suffix}_task.csv"
        try:
            _save_cm_csv(repA["cm"], cmA_path, ACTION_NAMES, ACTION_NAMES)
            _save_cm_csv(repT["cm"], cmT_path, TASK_NAMES, TASK_NAMES)
        except Exception as e:
            print(f"[warn] could not save CM for {split}/{scenario}{suffix}: {e}")

    if save_stream_csv and stream_rows is not None and len(stream_rows) > 0:
        stream_path = out_dir / f"stream_{split}_{scenario}{suffix}.csv"
        try:
            pd.DataFrame(stream_rows).to_csv(stream_path, index=False)
        except Exception as e:
            print(f"[warn] could not save stream CSV for {split}/{scenario}{suffix}: {e}")

    if save_preds and preds_pack is not None:
        preds_path = out_dir / f"preds_{split}_{scenario}{suffix}.npz"
        try:
            np.savez_compressed(preds_path, **preds_pack)
        except Exception as e:
            print(f"[warn] could not save preds NPZ for {split}/{scenario}{suffix}: {e}")

    return repA, repT, extras

# ============================================================
# ✅ Commit-threshold calibration on VAL (optimize false/1k REST)
# ============================================================
def _commit_events_stats_from_stream_rows(
    stream_rows: List[Dict[str, Any]],
    cfg: NeuroCommitCfg,
    stride_s: float = 0.25
):
    """
    Returns summary for calibration (paper-safe for balanced exports):
      - false_per_1k_rest: false commit events per 1000 REST windows (PRIMARY)
      - false_hr:          false commits per hour of REST (still returned, less stable on balanced exports)
      - flaps_pm:          state changes per minute
      - action_evt_pm:     commit events per minute during ACTION (avoid all-zero commits)
      - ct_sep:            mean(ct|ACTION) - mean(ct|REST)
      - n_rest:            number of REST windows (denominator for false_per_1k_rest)
    """
    if not stream_rows:
        return {
            "false_hr": float("inf"),
            "false_per_1k_rest": float("inf"),
            "flaps_pm": float("inf"),
            "action_evt_pm": 0.0,
            "ct_sep": 0.0,
            "n_rest": 0,
        }

    df = pd.DataFrame(stream_rows).copy()
    if ("center_time_s" not in df.columns) and ("onset_time_nearest_s" in df.columns) and ("dist_to_onset_s" in df.columns):
        ot = pd.to_numeric(df["onset_time_nearest_s"], errors="coerce")
        dd = pd.to_numeric(df["dist_to_onset_s"], errors="coerce")
        df["center_time_s"] = ot + dd

    key_cols = [c for c in ["subject_id", "task_code", "trial_id"] if c in df.columns]
    sort_candidates = ["center_time_s", "win_idx", "onset_time_nearest_s", "dist_to_onset_s", "_ord"]
    sort_key = next((c for c in sort_candidates if c in df.columns), None)
    if sort_key is None:
        df["_ord"] = np.arange(len(df), dtype=np.int64)
        sort_key = "_ord"

    # aggregates
    total_false_events = 0
    total_rest_dur_s = 0.0
    total_changes = 0
    total_dur_s = 0.0
    total_action_events = 0
    total_action_dur_s = 0.0

    # ✅ NEW: stable denominator for balanced exports
    n_rest = 0

    groups = [("__all__", df)] if len(key_cols) < 3 else list(df.groupby(key_cols))

    for _, g in groups:
        g = g.sort_values(sort_key)

        y = pd.to_numeric(g["y_action"], errors="coerce").fillna(0).astype(np.int64).to_numpy(copy=False)
        ct = pd.to_numeric(g["ct"], errors="coerce").fillna(0.0).to_numpy(np.float32, copy=False)

        filt = CommitDecisionFilter(
            thr_on=cfg.commit_thr_on,
            thr_off=cfg.commit_thr_off,
            dwell_windows=cfg.commit_dwell_windows,
            cooldown_windows=cfg.commit_cooldown_windows,
        )

        states = []
        events = np.zeros((len(ct),), dtype=np.int64)
        for t in range(len(ct)):
            st, ev = filt.step(float(ct[t]))
            states.append(st)
            events[t] = 1 if ev else 0

        states = np.asarray(states, dtype=object)
        rest = (y == 0)
        act  = (y == 1)

        # ✅ NEW: count REST windows
        n_rest += int(rest.sum())

        total_false_events += int(((events == 1) & rest).sum())
        total_rest_dur_s   += float(rest.sum()) * float(stride_s)

        total_action_events += int(((events == 1) & act).sum())
        total_action_dur_s  += float(act.sum()) * float(stride_s)

        total_changes += int((states[1:] != states[:-1]).sum()) if len(states) > 1 else 0
        total_dur_s   += float(len(states)) * float(stride_s)

    false_hr = (total_false_events / max(1e-9, total_rest_dur_s)) * 3600.0
    false_per_1k_rest = (total_false_events / max(1, n_rest)) * 1000.0  # ✅ NEW (primary)

    flaps_pm = (total_changes / max(1e-9, total_dur_s)) * 60.0
    action_evt_pm = (total_action_events / max(1e-9, total_action_dur_s)) * 60.0

    ct_all = pd.to_numeric(df["ct"], errors="coerce").to_numpy(np.float32, copy=False)
    y_all  = pd.to_numeric(df["y_action"], errors="coerce").fillna(0).astype(np.int64).to_numpy(copy=False)
    ct_rest = ct_all[y_all == 0]
    ct_act  = ct_all[y_all == 1]
    ct_sep = float(np.nanmean(ct_act) - np.nanmean(ct_rest)) if (ct_act.size and ct_rest.size) else 0.0

    return {
        "false_hr": float(false_hr),
        "false_per_1k_rest": float(false_per_1k_rest),
        "flaps_pm": float(flaps_pm),
        "action_evt_pm": float(action_evt_pm),
        "ct_sep": float(ct_sep),
        "n_rest": int(n_rest),
    }

def calibrate_commit_filter_on_val(
    model,
    va_loader,
    device: str,
    use_amp: bool,
    cfgM: NeuroCommitCfg,
    scenario_for_ct: str = "S0",
    commit_gate_threshold: float = 0.5,   # ✅ NEW name
):

    """
    ✅ Calibrates commit_thr_on/off/dwell on VAL so commit metrics are meaningful.
    IMPORTANT:
      - eval_one_split_fast must output HARD-GATED ct (ct==0 when pred_action==0)
      - We optimize false_per_1k_rest (paper-stable for balanced exports)
    """
    # collect stream_rows once (ct is already hard-gated by action_threshold inside eval)
    _, _, _, _, stream_rows = eval_one_split_fast(
        model=model,
        loader=va_loader,
        device=device,
        use_amp=use_amp,
        scenario=scenario_for_ct,

        # action_threshold is STILL required for action preds (but can just use the same threshold)
        action_threshold=float(commit_gate_threshold),

        # ✅ THIS is what gates ct
        commit_gate_threshold=float(commit_gate_threshold),

        cfgM=cfgM,
        robustness=None,
        save_preds=False,
        save_stream=True,
    )


    # grid search (broader since ct is hard-gated)
    thr_ons = np.linspace(0.30, 0.95, 14)
    dwells = [1, 2, 3, 4]
    best = None

    for thr_on in thr_ons:
        for dwell in dwells:
            tmp = NeuroCommitCfg(**asdict(cfgM))
            tmp.commit_thr_on = float(thr_on)
            tmp.commit_thr_off = float(max(0.02, thr_on - 0.20))
            tmp.commit_dwell_windows = int(dwell)
            tmp.commit_cooldown_windows = int(cfgM.commit_cooldown_windows)

            s = _commit_events_stats_from_stream_rows(stream_rows, tmp, stride_s=float(WIN_STRIDE_S))

            # ✅ paper-stable objective for balanced exports
            obj = (
                1.00 * float(s.get("false_per_1k_rest", float("inf")))  # primary
                + 0.05 * float(s.get("flaps_pm", float("inf")))         # stability
                - 0.10 * float(s.get("action_evt_pm", 0.0))             # avoid "all zero commits"
            )

            # hard penalty if too many false commits
            if float(s.get("false_per_1k_rest", float("inf"))) > 1.0:
                obj += 50.0 * (float(s["false_per_1k_rest"]) - 1.0)

            cand = {
                "obj": float(obj),
                "thr_on": float(tmp.commit_thr_on),
                "thr_off": float(tmp.commit_thr_off),
                "dwell": int(tmp.commit_dwell_windows),
                "stats": s,
            }
            if (best is None) or (cand["obj"] < best["obj"]):
                best = cand

    # apply best to cfgM
    if best is not None:
        cfgM.commit_thr_on = float(best["thr_on"])
        cfgM.commit_thr_off = float(best["thr_off"])
        cfgM.commit_dwell_windows = int(best["dwell"])

    return best


# ============================================================
# ✅ Per-scenario ACTION threshold tuning on VAL
# ============================================================
def tune_thresholds_by_scenario(
    model,
    va_loader,
    device: str,
    use_amp: bool,
    scenarios: List[str],
    objective: str,
    n_thresholds: int,
):
    thr_by = {}
    info_by = {}
    for sc in scenarios:
        info = tune_action_threshold_on_val_fast(
            model=model,
            va_loader=va_loader,
            device=device,
            use_amp=use_amp,
            scenario=sc,
            objective=objective,
            n_thresholds=n_thresholds,
        )
        thr_by[sc] = float(info["best_thr"])
        info_by[sc] = info
    return thr_by, info_by


# ============================================================
# RUN ALL FOLDS (LOSO) — skip missing folds gracefully
# ============================================================
import traceback

def fold_has_shards(dataset_dir: Path, exp_prefix: str, fold: int) -> bool:
    fold_dir = Path(dataset_dir) / f"{exp_prefix}_fold{int(fold)}"
    if not fold_dir.exists():
        return False
    for split in ["train", "val", "test"]:
        sd = fold_dir / split
        if not sd.exists():
            return False
        if len(list(sd.glob("*_shard_*.npz"))) == 0:
            return False
    return True

def get_folds_to_run() -> list:
    if AUTO_DETECT_FOLDS:
        folds = detect_folds(DATA_ROOT, SUP_EXP_PREFIX)
    else:
        folds = list(FOLDS_EXPLICIT)
    return sorted(set(int(f) for f in folds))

# ------------------------------------------------------------
# Optional: choose best epoch using robust val score
# ------------------------------------------------------------
VAL_SCORE_MODE = "S0"  # "S0" | "avg_S0_S3"
VAL_SCENARIOS_FOR_SCORE = ["S0","S1","S2","S3"]

def _val_score(repA_by_sc: Dict[str, dict]) -> float:
    if VAL_SCORE_MODE == "avg_S0_S3":
        vals = [float(repA_by_sc[s]["macro_f1"]) for s in VAL_SCENARIOS_FOR_SCORE if s in repA_by_sc]
        return float(np.mean(vals)) if vals else float(repA_by_sc["S0"]["macro_f1"])
    return float(repA_by_sc["S0"]["macro_f1"])
# ============================================================
# RUN ONE FOLD  (UPDATED: separate action thresholds vs commit-gate thresholds)
# ============================================================
def run_fold(fold: int):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    dev_type = "cuda" if device.startswith("cuda") else "cpu"

    seed_all(BASE_SEED + int(fold))

    out_dir = ensure_dir(ART_ROOT / TAG / f"fold{fold}")
    print(f"\n==================== FOLD {fold} ====================")
    print("[run] device:", device)
    print("[run] out_dir:", out_dir)

    tidy_rows = []
    perclass_rows = []

    runtime = {
        "tag": str(TAG),
        "fold": int(fold),
        "device": str(device),
        "amp": bool(USE_AMP),
        "num_workers": int(NUM_WORKERS),
        "cache_size": int(CACHE_SIZE),
        "bs_train": int(BS_TRAIN),
        "bs_eval": int(BS_EVAL),
        "ssl_bs_train": int(SSL_BS_TRAIN),
        "ssl_epochs": int(SSL_EPOCHS) if SSL_ENABLE else 0,
        "ft_epochs": int(EPOCHS),
        "thr_search_steps": int(THR_SEARCH_STEPS),
        "scenarios": list(ALL_SCENARIOS),
        "sup_exp_prefix": str(SUP_EXP_PREFIX),
        "ssl_exp_prefix": str(SSL_EXP_PREFIX),
        "train_scenario_mix": bool(TRAIN_SCENARIO_MIX),
        "dom_reg_enable": bool(DOM_REG_ENABLE),
        "dom_min_entropy": float(DOM_MIN_ENTROPY),
        "dom_reg_weight": float(DOM_REG_WEIGHT),
        "use_sample_weight": bool(USE_SAMPLE_WEIGHT),
        "val_score_mode": str(VAL_SCORE_MODE),
    }

    # ✅ hardware + reproducibility info (paper needs this)
    runtime["torch_version"] = str(torch.__version__)
    runtime["cuda_available"] = bool(torch.cuda.is_available())
    runtime["cuda_version"] = str(torch.version.cuda) if torch.cuda.is_available() else ""
    runtime["gpu_name"] = str(torch.cuda.get_device_name(0)) if torch.cuda.is_available() else ""
    runtime["amp_enabled"] = bool(USE_AMP)

    if dev_type == "cuda":
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    t_all0 = time.perf_counter()

    # -------------------------
    # Loaders (SUP fold)
    # -------------------------
    tr_loader, va_loader, te_loader, Ce, Cm, Ct, _, rel_dim_infer, use_feats, sampler_enabled, batch_sampler = make_loaders(
        dataset_dir=DATA_ROOT,
        exp_prefix=SUP_EXP_PREFIX,
        fold=fold,
        bs_train=BS_TRAIN,
        bs_eval=BS_EVAL,
        num_workers=NUM_WORKERS,
        label_frac=1.0,
        seed=BASE_SEED + fold,
        use_p55_features=USE_P55_FEATURES,
        feature_pattern=FEATURE_PATTERN,
        strict_feature_align=True,
        use_action_sampler=USE_ACTION_SAMPLER,
        sampler_mode=SAMPLER_MODE,
        rel_dim=None,
        cache_size=CACHE_SIZE,
        device=device,
    )
    print(f"[data] fold={fold} Ce={Ce} Cm={Cm} Ct={Ct} rel_dim={rel_dim_infer} feats={use_feats} sampler={sampler_enabled}")

    # ✅ dataset sizes (optional but useful for paper)
    runtime["n_train_windows"] = int(len(tr_loader.dataset))
    runtime["n_val_windows"] = int(len(va_loader.dataset))
    runtime["n_test_windows"] = int(len(te_loader.dataset))

    runtime.update({
        "Ce": int(Ce), "Cm": int(Cm), "Ct": int(Ct),
        "rel_dim": int(rel_dim_infer),
        "use_feats": bool(use_feats),
        "sampler_enabled": bool(sampler_enabled),
    })

    if batch_sampler is not None:
        batch_sampler.set_epoch(0)

    # -------------------------
    # Model
    # -------------------------
    cfgM = NeuroCommitCfg(
        d_model=160,
        patch=SSL_PATCH_SIZE,
        rel_dim=int(rel_dim_infer),
        use_p55_features=bool(use_feats),
        feat_dim_psd=96, feat_dim_emg=24, feat_dim_mask=3,
        feat_z_scale=0.25,
        feat_token_scale=0.25,
    )
    model = NeuroCommitM3(Ce=Ce, Cm=Cm, Ct=Ct, cfg=cfgM, num_task=5).to(device)
    runtime["param_count"] = int(count_params(model))

    # -------------------------
    # SSL pretrain (optional)
    # -------------------------
    t_ssl0 = time.perf_counter()
    if SSL_ENABLE:
        ssl_tr, _, _, Ce2, Cm2, Ct2, _, rel_dim_ssl, use_feats_ssl, _, _ = make_loaders(
            dataset_dir=DATA_ROOT,
            exp_prefix=SSL_EXP_PREFIX,
            fold=fold,
            bs_train=SSL_BS_TRAIN,
            bs_eval=BS_EVAL,
            num_workers=NUM_WORKERS,
            label_frac=1.0,
            seed=BASE_SEED + 999 + fold,
            use_p55_features=False,  # keep SSL clean
            feature_pattern=FEATURE_PATTERN,
            strict_feature_align=True,
            use_action_sampler=False,
            sampler_mode="none",
            rel_dim=None,
            cache_size=CACHE_SIZE,
            device=device,
        )
        print(f"[ssl data] Ce={Ce2} Cm={Cm2} Ct={Ct2} rel_dim={rel_dim_ssl} feats={use_feats_ssl}")

        opt_ssl = torch.optim.AdamW(model.parameters(), lr=SSL_LR, weight_decay=SSL_WD)
        scaler_ssl = torch.amp.GradScaler("cuda", enabled=(USE_AMP and dev_type == "cuda")) if dev_type == "cuda" else None

        for ep in range(1, SSL_EPOCHS + 1):
            stats = train_one_epoch_ssl(
                model=model,
                loader=ssl_tr,
                opt=opt_ssl,
                scaler=scaler_ssl,
                device=device,
                patch=SSL_PATCH_SIZE,
                temp=SSL_TEMP,
                lambda_nce=SSL_LAMBDA_NCE,
                lambda_rec=SSL_LAMBDA_REC,
                mask_patch_frac=SSL_MASK_PATCH_FRAC,
                use_amp=USE_AMP,
                grad_clip=1.0,
            )
            print(f"[SSL] ep {ep:02d}/{SSL_EPOCHS} | loss={stats['ssl_loss']:.4f} nce={stats['ssl_nce']:.4f} rec={stats['ssl_rec']:.4f}")

        torch.save({"cfg": asdict(cfgM), "state_dict": model.state_dict()}, out_dir / "ssl_last.pt")
        print("[SSL] saved:", out_dir / "ssl_last.pt")

    if dev_type == "cuda":
        torch.cuda.synchronize()
    runtime["ssl_time_s"] = float(time.perf_counter() - t_ssl0)

    # -------------------------
    # Supervised FT
    # -------------------------
    w_task, task_counts = compute_task_class_weights_from_shards(tr_loader.dataset, num_task=5)
    w_task_t = torch.tensor(w_task, dtype=torch.float32, device=device)
    print("[task weights] counts:", task_counts, "w_task:", w_task)

    runtime["task_counts"] = [int(x) for x in np.asarray(task_counts).reshape(-1).tolist()]
    runtime["task_weights"] = [float(x) for x in np.asarray(w_task).reshape(-1).tolist()]

    opt = torch.optim.AdamW(model.parameters(), lr=2e-4, weight_decay=1e-2)
    scaler_ft = torch.amp.GradScaler("cuda", enabled=(USE_AMP and dev_type == "cuda")) if dev_type == "cuda" else None

    best_val = -1e9
    best_path = out_dir / "best.pt"

    # we keep a single thr for training-time val (fast)
    t_ft0 = time.perf_counter()
    for ep in range(1, EPOCHS + 1):
        if batch_sampler is not None:
            batch_sampler.set_epoch(ep)

        tr_stats = train_one_epoch(
            model=model,
            loader=tr_loader,
            opt=opt,
            scaler=scaler_ft,
            device=device,
            w_task=w_task_t,
            alpha_task=ALPHA_TASK,
            use_sample_weight=USE_SAMPLE_WEIGHT,
            grad_clip=1.0,
            use_amp=USE_AMP,
        )

        # tune ACTION thr on VAL for S0 (fast)
        thr_info = tune_action_threshold_on_val_fast(
            model=model,
            va_loader=va_loader,
            device=device,
            use_amp=USE_AMP,
            scenario="S0",
            objective=THR_OBJECTIVE,
            n_thresholds=THR_SEARCH_STEPS,
        )
        thr_s0 = float(thr_info["best_thr"])

        # evaluate val score (S0 or avg S0..S3)
        repA_by = {}
        for sc in (["S0"] if VAL_SCORE_MODE == "S0" else VAL_SCENARIOS_FOR_SCORE):
            repA_va, _, _, _ = eval_one_split_fast(
                model=model,
                loader=va_loader,
                device=device,
                use_amp=USE_AMP,
                scenario=sc,
                action_threshold=thr_s0,
                commit_gate_threshold=thr_s0,   # ✅ keep gating consistent during FT-val
                cfgM=cfgM,
                robustness=None,
                save_preds=False,
                save_stream=False,
            )
            repA_by[sc] = repA_va

        score = _val_score(repA_by)
        print(
            f"[FT] ep {ep:02d}/{EPOCHS} | train_loss={tr_stats['train_loss']:.4f} | "
            f"val_score={score:.4f} | thr(S0)={thr_s0:.3f} "
            f"| domH={tr_stats.get('dom_entropy_mean', float('nan')):.3f} domL={tr_stats.get('dom_loss_mean', float('nan')):.4f}"
        )

        if score > best_val:
            best_val = score
            torch.save(model.state_dict(), best_path)
            with open(out_dir / "best_meta.json", "w") as f:
                json.dump({"cfg": asdict(cfgM), "thr_s0": float(thr_s0)}, f, indent=2)
            print("[FT] saved best:", best_path)

    if dev_type == "cuda":
        torch.cuda.synchronize()
    runtime["ft_time_s"] = float(time.perf_counter() - t_ft0)

    if not best_path.exists():
        raise RuntimeError(f"best.pt was not saved: {best_path}")

    # -------------------------
    # Load best
    # -------------------------
    try:
        sd = torch.load(best_path, map_location=device, weights_only=True)
    except TypeError:
        sd = torch.load(best_path, map_location=device)

    model.load_state_dict(sd)
    with open(out_dir / "best_meta.json", "r") as f:
        meta = json.load(f)
    thr_s0_best = float(meta.get("thr_s0", 0.5))

    print("[EVAL] loaded best:", best_path, "thr(S0)=", thr_s0_best)

    runtime["best_val_score"] = float(best_val)
    runtime["thr_s0_best"] = float(thr_s0_best)

    # =========================================================
    # ✅ THRESHOLDS (FIXED): separate ACTION thresholds vs COMMIT-GATE thresholds
    # =========================================================

    # 1) ACTION thresholds (for macroF1 reporting)
    thr_action_by_sc, thr_info_by_sc = tune_thresholds_by_scenario(
        model=model,
        va_loader=va_loader,
        device=device,
        use_amp=USE_AMP,
        scenarios=ALL_SCENARIOS,
        objective=THR_OBJECTIVE,
        n_thresholds=THR_SEARCH_STEPS,
    )

    # 2) COMMIT-GATE thresholds (only for ct gating)
    thr_commit_gate_by_sc = dict(thr_action_by_sc)
    for sc, lam in [("S2", 3.0), ("S6", 5.0)]:
        thr_commit_gate_by_sc[sc] = float(tune_action_threshold_commit_safe(
            model=model,
            va_loader=va_loader,
            device=device,
            use_amp=USE_AMP,
            scenario=sc,
            n_thresholds=THR_SEARCH_STEPS,
            lambda_fpr=lam,
            thr_cap=0.97,  # optional
        ))


    print("[thr_action_by_sc]", thr_action_by_sc)
    print("[thr_commit_gate_by_sc]", thr_commit_gate_by_sc)

    runtime["thr_action_by_scenario"] = {k: float(v) for k, v in thr_action_by_sc.items()}
    runtime["thr_commit_gate_by_scenario"] = {k: float(v) for k, v in thr_commit_gate_by_sc.items()}

    # -------------------------
    # ✅ Commit filter calibration on VAL (use COMMIT-GATE threshold for gating ct)
    # -------------------------
    best_commit = calibrate_commit_filter_on_val(
        model=model,
        va_loader=va_loader,
        device=device,
        use_amp=USE_AMP,
        cfgM=cfgM,
        scenario_for_ct="S0",
        commit_gate_threshold=float(thr_commit_gate_by_sc.get("S0", thr_s0_best)),
    )

    # extra safety clamp (optional)
    cfgM.commit_thr_on = max(cfgM.commit_thr_on, 0.80)
    cfgM.commit_thr_off = max(0.02, cfgM.commit_thr_on - 0.20)
    cfgM.commit_dwell_windows = max(cfgM.commit_dwell_windows, 2)

    runtime["commit_calibration"] = best_commit
    runtime["commit_thr_on"] = float(cfgM.commit_thr_on)
    runtime["commit_thr_off"] = float(cfgM.commit_thr_off)
    runtime["commit_dwell_windows"] = int(cfgM.commit_dwell_windows)
    print(
        f"[commit calib] thr_on={cfgM.commit_thr_on:.2f} thr_off={cfgM.commit_thr_off:.2f} "
        f"dwell={cfgM.commit_dwell_windows} stats={best_commit['stats'] if best_commit else None}"
    )

    # -------------------------
    # Final EVAL (VAL + TEST) across S0–S6 using BOTH thresholds
    #   - thr_action: for ACTION metrics (macroF1, etc.)
    #   - thr_gate:   ONLY for gating ct / commit metrics
    #   - cap thr_gate for S2/S6 to avoid "never commits"
    # -------------------------
    t_ev0 = time.perf_counter()

    THR_GATE_CAP_S2S6 = 0.95  # set 0.97 if you want less strict cap

    for split_name, loader in [("val", va_loader), ("test", te_loader)]:
        for sc in ALL_SCENARIOS:
            thr_action = float(thr_action_by_sc.get(sc, thr_s0_best))
            thr_gate   = float(thr_commit_gate_by_sc.get(sc, thr_action))

            # ✅ cap commit-gate threshold for the harsh missing-modality scenarios
            if sc in ("S2", "S6"):
                thr_gate = min(thr_gate, float(THR_GATE_CAP_S2S6))

            repA, repT, extras = eval_log_save_split(
                model=model,
                loader=loader,
                device=device,
                use_amp=USE_AMP,
                fold=fold,
                split=split_name,
                scenario=sc,
                action_threshold=thr_action,            # ✅ for macroF1
                commit_gate_threshold=thr_gate,         # ✅ for ct gating / commit metrics
                cfgM=cfgM,
                out_dir=out_dir,
                tidy_rows=tidy_rows,
                perclass_rows=perclass_rows,
                tag=TAG,
                label_frac=1.0,
                robustness=None,
                save_preds=SAVE_PREDS,
                save_stream_csv=True,
                save_cm_csv=True,
            )

            print(
                f"[{split_name.upper()} {sc}] thrA={thr_action:.3f} thrCT={thr_gate:.3f} "
                f"action_macroF1={repA['macro_f1']:.4f} "
                f"false_per_1k_rest={float(extras.get('commit_false_per_1000_rest', float('nan'))):.2f} "
                f"false_events={int(extras.get('commit_false_events', -1))} "
                f"state_changes={int(extras.get('commit_state_changes', -1))} "
                f"ct_p50={float(extras.get('ct_p50', float('nan'))):.3f}"
            )

    if dev_type == "cuda":
        torch.cuda.synchronize()
    runtime["eval_time_s"] = float(time.perf_counter() - t_ev0)


    # Optional robustness eval
    if DO_R3R4_EVAL:
        for split_name, loader in [("test", te_loader)]:
            for sc in ALL_SCENARIOS:
                thr_action = float(thr_action_by_sc.get(sc, thr_s0_best))
                thr_gate   = float(thr_commit_gate_by_sc.get(sc, thr_action))
                for rob in ["R3", "R4"]:
                    repA, repT, extras = eval_log_save_split(
                        model=model,
                        loader=loader,
                        device=device,
                        use_amp=USE_AMP,
                        fold=fold,
                        split=split_name,
                        scenario=sc,
                        action_threshold=thr_action,
                        commit_gate_threshold=thr_gate,
                        cfgM=cfgM,
                        out_dir=out_dir,
                        tidy_rows=tidy_rows,
                        perclass_rows=perclass_rows,
                        tag=TAG,
                        label_frac=1.0,
                        robustness=rob,
                        save_preds=False,
                        save_stream_csv=False,
                        save_cm_csv=False,
                    )
                    print(f"[{split_name.upper()} {sc} {rob}] thrA={thr_action:.3f} thrCT={thr_gate:.3f} action_macroF1={repA['macro_f1']:.4f}")

    if dev_type == "cuda":
        torch.cuda.synchronize()
    runtime["eval_time_s"] = float(time.perf_counter() - t_ev0)

    # -------------------------
    # Latency + Peak memory
    # -------------------------
    try:
        sample_batch = next(iter(te_loader))
        for k, v in list(sample_batch.items()):
            if torch.is_tensor(v) and v.ndim >= 1 and v.shape[0] > 1:
                sample_batch[k] = v[:1]
        runtime["infer_latency_ms"] = float(measure_infer_latency_ms(
            model=model, sample_batch=sample_batch, device=device,
            iters=200, warmup=20, use_amp=USE_AMP
        ))
    except Exception as e:
        runtime["infer_latency_ms"] = float("nan")
        runtime["infer_latency_err"] = str(e)

    if dev_type == "cuda":
        runtime["peak_mem_gb"] = float(torch.cuda.max_memory_allocated() / (1024**3))

    runtime["total_time_s"] = float(time.perf_counter() - t_all0)

    # -------------------------
    # Save fold summary + CSVs + runtime JSON
    # -------------------------
    summary = {
        "fold": int(fold),
        "best_val_score": float(best_val),
        "thr_s0_best": float(thr_s0_best),
        "runtime": runtime,
        "cfgM": asdict(cfgM),
    }
    with open(out_dir / f"summary_fold{fold}.json", "w") as f:
        json.dump(summary, f, indent=2)

    pd.DataFrame(tidy_rows).to_csv(out_dir / f"evals_tidy_fold{fold}.csv", index=False)
    pd.DataFrame(perclass_rows).to_csv(out_dir / f"perclass_fold{fold}.csv", index=False)

    save_runtime_json(out_dir, fold, runtime)

    print("[done] wrote:", out_dir / f"summary_fold{fold}.json")
    print("[done] wrote:", out_dir / f"evals_tidy_fold{fold}.csv")
    print("[done] wrote:", out_dir / f"perclass_fold{fold}.csv")
    print("[done] wrote:", out_dir / f"runtime_fold{fold}.json")


# ============================================================
# RUN ALL FOLDS — skip missing
# ============================================================
def run_all_folds(skip_existing: bool = True):
    folds = get_folds_to_run()
    print("[all-folds] candidate folds:", folds)

    ok, skipped, failed = [], [], []

    for fold in folds:
        if not fold_has_shards(DATA_ROOT, SUP_EXP_PREFIX, fold):
            print(f"[skip] fold{fold}: missing SUP shards under {SUP_EXP_PREFIX}_fold{fold}")
            skipped.append((fold, "missing_sup"))
            continue

        if SSL_ENABLE and (not fold_has_shards(DATA_ROOT, SSL_EXP_PREFIX, fold)):
            print(f"[skip] fold{fold}: missing SSL shards under {SSL_EXP_PREFIX}_fold{fold}")
            skipped.append((fold, "missing_ssl"))
            continue

        out_dir = ART_ROOT / TAG / f"fold{fold}"
        if skip_existing and (out_dir / f"evals_tidy_fold{fold}.csv").exists() and (out_dir / f"runtime_fold{fold}.json").exists():
            print(f"[skip] fold{fold}: artifacts already exist at {out_dir}")
            skipped.append((fold, "already_done"))
            continue

        try:
            run_fold(fold)
            ok.append(fold)
        except Exception as e:
            print(f"[FAIL] fold{fold}: {repr(e)}")
            traceback.print_exc()
            failed.append((fold, repr(e)))

    print("\n==================== DONE ====================")
    print("[all-folds] OK:", ok)
    print("[all-folds] SKIPPED:", skipped)
    print("[all-folds] FAILED:", failed)
    return ok, skipped, failed
# ============================================================
# Post-run: Export per-fold metric "matrix" for later Wilcoxon
# ============================================================

def export_metric_matrix_for_tag(
    tag: str,
    out_csv: Path,
    split: str = "test",
    scenarios: Optional[List[str]] = None,
    robustness: str = "",
    metrics: Optional[List[str]] = None,
):
    scenarios = scenarios or ["S0","S1","S2","S3","S4","S5","S6"]
    metrics = metrics or [
        # ACTION
        "action_macroF1",
        "action_bal_acc",
        "action_AUPRC",
        "action_rest_fpr",
        # TASK
        "task_macroF1",
        "task_bal_acc",
        "task_AUPRC",
        # COMMIT
        "commit_false_per_1000_rest",
        "commit_false_per_hour_rest",
        "commit_false_per_hour_total",
        "commit_flaps_per_min",
        # CT summaries
        "ct_p50",
        "time_to_commit_mean_s",
        # thresholds (useful for debugging / reporting)
        "action_threshold",
        "commit_gate_threshold",
    ]

    root = ART_ROOT / str(tag)
    rows: List[Dict[str, Any]] = []

    for fold_dir in sorted(root.glob("fold*")):
        try:
            fold = int(fold_dir.name.replace("fold", ""))
        except Exception:
            continue

        p = fold_dir / f"evals_tidy_fold{fold}.csv"
        if not p.exists():
            continue

        df = pd.read_csv(p)
        if "robustness" in df.columns:
            df["robustness"] = df["robustness"].fillna("")
        else:
            df["robustness"] = ""

        for sc in scenarios:
            d = df[(df["split"] == split) & (df["scenario"] == sc) & (df["robustness"] == robustness)]
            if len(d) == 0:
                continue

            row0 = d.iloc[0]
            r = {
                "tag": str(tag),
                "fold": int(fold),
                "split": str(split),
                "scenario": str(sc),
                "robustness": str(robustness),
            }
            for m in metrics:
                r[m] = float(row0[m]) if (m in d.columns and pd.notna(row0[m])) else float("nan")
            rows.append(r)

    out_csv = Path(out_csv)
    out_csv.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(out_csv, index=False)
    print("[metric-matrix] wrote:", out_csv)


# ============================================================
# Run folds FIRST, then export matrices (post-run, complete)
# ============================================================

if __name__ == "__main__":
    # ---- run folds first ----
    ok, skipped, failed = run_all_folds(skip_existing=True)

    # ---- then export matrices (base: no R3/R4) ----
    export_metric_matrix_for_tag(
        tag=TAG,
        out_csv=ART_ROOT / TAG / "metric_matrix_test.csv",
        split="test",
        scenarios=ALL_SCENARIOS,
        robustness="",   # base (no R3/R4)
    )

    export_metric_matrix_for_tag(
        tag=TAG,
        out_csv=ART_ROOT / TAG / "metric_matrix_val.csv",
        split="val",
        scenarios=ALL_SCENARIOS,
        robustness="",   # base (no R3/R4)
    )

    # Optional: also export R3/R4 matrices if you enable DO_R3R4_EVAL later
    export_metric_matrix_for_tag(
        tag=TAG,
        out_csv=ART_ROOT / TAG / "metric_matrix_test_R3.csv",
        split="test",
        scenarios=ALL_SCENARIOS,
        robustness="R3",
    )
    export_metric_matrix_for_tag(
        tag=TAG,
        out_csv=ART_ROOT / TAG / "metric_matrix_test_R4.csv",
        split="test",
        scenarios=ALL_SCENARIOS,
        robustness="R4",
    )

